
# المرحلة الثانية: بناء نظام الاسترجاع وتقييمه
## Retrieval Pipeline and Evaluation

### 1. الهدف
بناء نظام استرجاع ثنائي اللغة (العربية والإنجليزية) لاسترجاع النصوص والأدلة القانونية ذات الصلة بنظام حماية البيانات الشخصية السعودي (PDPL)، لدعم الإجابة عن الأسئلة وتحليل الامتثال لاحقًا.

### 2. المنهجية
سننفذ ونقارن أربع طرق:
1. الاسترجاع بالكلمات المفتاحية (BM25).
2. الاسترجاع الدلالي (Dense Retrieval).
3. الاسترجاع الهجين باستخدام دمج الرتب (RRF).
4. إعادة ترتيب المقاطع باستخدام Cross-Encoder.

### 3. التقييم
سنستخدم مجموعة الأسئلة المرجعية (Gold QA)، بعد التحقق من أهلية الأسئلة وصحة ربطها بالأدلة القانونية.

المقاييس الأساسية:
- Recall@5: مدى نجاح النظام في استرجاع الأدلة الصحيحة ضمن أول خمسة مقاطع.
- MRR@10: مدى جودة ترتيب أول دليل صحيح ضمن أول عشرة مقاطع.

### 4. نطاق المرحلة
تركز هذه المرحلة على الاسترجاع وترتيب الأدلة. أما توليد الإجابات وتحليل السياسات واكتشاف التعارضات والفجوات والأدلة الناقصة، فستُنفذ في المراحل التالية.

### 5. توثيق النتائج
سنوضح المنهجية والنتائج الفعلية والاستنتاجات والقيود لكل تجربة، ولن نحدد الطريقة الأفضل قبل مقارنة نتائج التقييم.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
 "rank_bm25": "rank-bm25",
 "sentence_transformers": "sentence-transformers",
 "transformers": "transformers",
 "tqdm": "tqdm",
 "pandas": "pandas",
 "numpy": "numpy",
 "matplotlib": "matplotlib",
 "seaborn": "seaborn",
}

missing_packages = [
 pip_name
 for module_name, pip_name in required_packages.items()
 if importlib.util.find_spec(module_name) is None
]

print("المكتبات الناقصة:", missing_packages or "لا توجد")

subprocess.check_call(
 [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
) if missing_packages else print("لا حاجة إلى التثبيت.")

still_missing = [
 module_name
 for module_name in required_packages
 if importlib.util.find_spec(module_name) is None
]

assert not still_missing, f"تعذر العثور على المكتبات: {still_missing}"
print("اكتمل فحص المكتبات بنجاح.")


المكتبات الناقصة: ['rank-bm25']
اكتمل فحص المكتبات بنجاح.


In [2]:
#  FAISS 
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("faiss") is None:
 subprocess.check_call([
 sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"
 ])

import faiss

print("إصدار FAISS:", faiss.__version__)
print("FAISS جاهز.")


إصدار FAISS: 1.15.1
FAISS جاهز.


In [3]:
from pathlib import Path
import torch

# فحص GPU؛ سيعمل المشروع على CPU إذا لم يتوفر
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA متاح:", torch.cuda.is_available())
print("الجهاز:", DEVICE)

if torch.cuda.is_available():
 print("اسم GPU:", torch.cuda.get_device_name(0))

# المسارات المطلوبة كما وردت، بما فيها الاسم الفعلي artical
BASE = Path("/kaggle/input/datasets/khadijah2000")

FILES = {
 "ar_fixed": BASE / "pdpl-chunking-comparison-datasets/fixed_size_chunks.json",
 "ar_article": BASE / "pdpl-chunking-comparison-datasets/article_section_chunks.json",
 "en_fixed": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json",
 "en_article": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json",
 "gold_qa": BASE / "pdpl-bilingual-evaluation-dataset/gold_qa (2).json",
}

missing_paths = [str(path) for path in FILES.values() if not path.is_file()]

assert not missing_paths, (
 "الملفات التالية غير موجودة. تحققي من إضافة مجموعة البيانات إلى Kaggle:\n"
 + "\n".join(missing_paths)
)

print("\nتم العثور على جميع الملفات:")
for name, path in FILES.items():
 print(f"{name}: {path}")


CUDA متاح: True
الجهاز: cuda
اسم GPU: Tesla T4

تم العثور على جميع الملفات:
ar_fixed: /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets/fixed_size_chunks.json
ar_article: /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets/article_section_chunks.json
en_fixed: /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json
en_article: /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json
gold_qa: /kaggle/input/datasets/khadijah2000/pdpl-bilingual-evaluation-dataset/gold_qa (2).json


In [4]:
import json
import pandas as pd

# تحميل الملفات الموجودة في المسارات التي فحصناها
RAW = {
 name: json.loads(path.read_text(encoding="utf-8"))
 for name, path in FILES.items()
}

# وصف البنية العليا لكل ملف دون افتراض أن الجذر قائمة أو قاموس
inspection_rows = []

for name, obj in RAW.items():
 is_dict = isinstance(obj, dict)
 is_list = isinstance(obj, list)

 top_level_details = (
 {
 key: {
 "type": type(value).__name__,
 "length": len(value) if isinstance(value, (list, dict)) else None,
 "first_item_keys": (
 list(value[0].keys())
 if isinstance(value, list)
 and len(value) > 0
 and isinstance(value[0], dict)
 else None
 ),
 }
 for key, value in obj.items()
 }
 if is_dict
 else None
 )

 sample = obj[0] if is_list and len(obj) > 0 else obj
 sample_keys = list(sample.keys()) if isinstance(sample, dict) else None

 inspection_rows.append({
 "file": name,
 "root_type": type(obj).__name__,
 "root_length": len(obj) if isinstance(obj, (list, dict)) else None,
 "root_keys": list(obj.keys()) if is_dict else None,
 "top_level_details": top_level_details,
 "sample_record_keys": sample_keys,
 "sample_preview": json.dumps(sample, ensure_ascii=False)[:1000],
 })

structure_report = pd.DataFrame(inspection_rows)
display(structure_report)


,file,root_type,root_length,root_keys,top_level_details,sample_record_keys,sample_preview
0,ar_fixed,list,241,None,None,"[id, document_id, language, article_number, ar...","{""id"": ""pdpl_ar_001"", ""document_id"": ""pdpl"", ""..."
1,ar_article,list,196,None,None,"[id, document_id, language, article_number, ar...","{""id"": ""pdpl_ar_001"", ""document_id"": ""pdpl"", ""..."
2,en_fixed,list,181,None,None,"[chunk_id, document_id, language, chunking, ch...","{""chunk_id"": ""accreditation_certificates_fixed..."
3,en_article,list,252,None,None,"[chunk_id, parent_id, document_id, language, u...","{""chunk_id"": ""pdpl_001_chunk_01"", ""parent_id"":..."
4,gold_qa,list,90,None,None,"[question_id, language, question, reference_an...","{""question_id"": ""AR001"", ""language"": ""ar"", ""qu..."


In [5]:
import pandas as pd

schema_rows = [
 {
 "file": file_name,
 "field": field,
 "records_with_value": sum(
 isinstance(record, dict)
 and field in record
 and record[field] is not None
 for record in records
 ),
 "total_records": len(records),
 "sample_value": repr(next(
 (
 record.get(field)
 for record in records
 if isinstance(record, dict)
 and record.get(field) is not None
 ),
 None
 ))[:220],
 }
 for file_name, records in RAW.items()
 if isinstance(records, list)
 for field in sorted({
 field
 for record in records
 if isinstance(record, dict)
 for field in record.keys()
 })
]

schema_report = pd.DataFrame(schema_rows)
display(schema_report)


,file,field,records_with_value,total_records,sample_value
0,ar_fixed,article_number,172,241,1
1,ar_fixed,article_title,241,241,'المادة الأولى:'
2,ar_fixed,chunk_count,241,241,4
3,ar_fixed,chunk_id,241,241,'pdpl_ar_001__fixed_size__001'
4,ar_fixed,chunk_index,241,241,1
...,...,...,...,...,...
109,gold_qa,revision_action,90,90,'الاحتفاظ بالسؤال الأصلي بعد مراجعة أولية'
110,gold_qa,revision_reason,90,90,'لا يوجد سبب موضوعي واضح لتغيير الصياغة قبل ال...
111,gold_qa,source_document,85,90,'نظام حماية البيانات الشخصية'
112,gold_qa,source_id_original,75,90,'PDPL'


In [6]:
# عرض جميع أسماء الحقول، ثم معاينة الحقول المهمة دون اختصار الصفوف
field_lists = "\n\n".join(
 file_name + ":\n " + ", ".join(
 schema_report.loc[schema_report["file"] == file_name, "field"].tolist()
 )
 for file_name in RAW.keys()
)

print(field_lists)

important_fields = schema_report[
 schema_report["field"].str.contains(
 r"text|chunk|article|document|reference|answer|eligible|corpus|language|question|source|url|id",
 case=False,
 regex=True,
 na=False
 )
]

pd.set_option("display.max_rows", None)
display(important_fields)


ar_fixed:
 article_number, article_title, chunk_count, chunk_id, chunk_index, chunking_strategy, clean_text, content_selector, document_id, document_title, extracted_at_utc, extraction_status, heading_level, html_sha256, id, language, processed_at_utc, quality_flags, search_text, section_id, section_order, source_input_file, source_record_id, source_type, source_url, text, text_sha256, token_count

ar_article:
 article_number, article_title, chunk_count, chunk_id, chunk_index, chunking_strategy, clean_text, content_selector, document_id, document_title, extracted_at_utc, extraction_status, heading_level, html_sha256, id, language, processed_at_utc, quality_flags, search_text, section_id, section_order, source_input_file, source_record_id, source_type, source_url, text, text_sha256, token_count

en_fixed:
 article_number, article_title, char_end, char_start, chunk_id, chunk_index, chunking, clean_text, covered_article_numbers, document_id, extraction_status, language, search_text, sourc

,file,field,records_with_value,total_records,sample_value
0,ar_fixed,article_number,172,241,1
1,ar_fixed,article_title,241,241,'المادة الأولى:'
2,ar_fixed,chunk_count,241,241,4
3,ar_fixed,chunk_id,241,241,'pdpl_ar_001__fixed_size__001'
4,ar_fixed,chunk_index,241,241,1
5,ar_fixed,chunking_strategy,241,241,'fixed_size'
6,ar_fixed,clean_text,241,241,'لغرض تطبيق هذا النظام، يُقصد بالألفاظ والعبار...
8,ar_fixed,document_id,241,241,'pdpl'
9,ar_fixed,document_title,13,241,'سياسة حق الحصول على المعلومة | وزارة الموارد ...
14,ar_fixed,id,241,241,'pdpl_ar_001'


In [7]:
# مقارنة حقول النص لتحديد النص المناسب للفهرسة دون تغيير البيانات
text_field_audit = pd.DataFrame([
 {
 "dataset": dataset_name,
 "chunk_id": record.get("chunk_id"),
 "text_chars": (
 len(record["text"])
 if isinstance(record.get("text"), str)
 else None
 ),
 "clean_text_chars": (
 len(record["clean_text"])
 if isinstance(record.get("clean_text"), str)
 else None
 ),
 "search_text_chars": (
 len(record["search_text"])
 if isinstance(record.get("search_text"), str)
 else None
 ),
 "text_equals_clean": (
 record["text"] == record["clean_text"]
 if isinstance(record.get("text"), str)
 and isinstance(record.get("clean_text"), str)
 else None
 ),
 }
 for dataset_name in ("ar_fixed", "ar_article", "en_fixed", "en_article")
 for record in RAW[dataset_name]
])

text_field_summary = (
 text_field_audit
 .groupby("dataset")
 .agg(
 records=("chunk_id", "size"),
 records_with_text=("text_chars", "count"),
 records_with_clean_text=("clean_text_chars", "count"),
 text_mean_chars=("text_chars", "mean"),
 text_max_chars=("text_chars", "max"),
 clean_mean_chars=("clean_text_chars", "mean"),
 clean_max_chars=("clean_text_chars", "max"),
 search_mean_chars=("search_text_chars", "mean"),
 text_clean_identical_fraction=("text_equals_clean", "mean"),
 )
 .reset_index()
)

display(text_field_summary.round(3))


,dataset,records,records_with_text,records_with_clean_text,text_mean_chars,text_max_chars,clean_mean_chars,clean_max_chars,search_mean_chars,text_clean_identical_fraction
0,ar_article,196,196,196,749.444,2100.0,749.444,2100,746.633,1.0
1,ar_fixed,241,241,241,655.842,1319.0,655.842,1319,654.772,1.0
2,en_article,252,0,252,NaN,NaN,733.425,1199,733.425,NaN
3,en_fixed,181,0,181,NaN,NaN,1166.514,1199,1166.514,NaN


In [8]:
# تقطيع مادة التعريفات (المادة الأولى) إلى «تعريف لكل بند» قبل الفهرسة
# يطابق تمامًا تقطيع المرحلة 1 (نفس التسمية pdpl_{lang}_001__defsplit__NNN)
# حتى تتطابق المقاطع بين فهرس المرحلة 2 وسجلّ المرحلة 3. إضافي: نُبقي الأصل.
import re as _re, copy as _copy

def _split_definitions(t):
    parts = _re.split(r'(?<!\d)(\d{1,2})\s*-\s*', t or "")
    out = []
    for i in range(1, len(parts) - 1, 2):
        num, body = parts[i], parts[i + 1].strip()
        if len(body) > 15:
            out.append(f"{num}- {body}".strip())
    return out

for _dataset_name in ("ar_article", "en_article"):
    _records = RAW[_dataset_name]
    _lang = "ar" if _dataset_name.startswith("ar") else "en"
    _a1 = sorted(
        [r for r in _records if r.get("document_id") == "pdpl" and str(r.get("article_number")) == "1"],
        key=lambda r: (r.get("chunk_index", 0), str(r.get("chunk_id", ""))),
    )
    if not _a1:
        print(f"{_dataset_name}: لا توجد مادة أولى — تخطٍّ")
        continue
    _full = " ".join(
        (r.get("search_text") or r.get("clean_text") or r.get("text") or "") for r in _a1
    )
    _defs = _split_definitions(_full)
    _tmpl = _copy.deepcopy(_a1[0])
    _new = []
    for _k, _body in enumerate(_defs, 1):
        _c = _copy.deepcopy(_tmpl)
        _cid = f"pdpl_{_lang}_001__defsplit__{_k:03d}"
        for _fld in ("text", "clean_text", "search_text"):
            _c[_fld] = _body
        _c["chunk_id"] = _cid
        _c["id"] = _cid
        _c["article_number"] = "1"
        _c["chunk_index"] = _k
        _c["chunk_count"] = len(_defs)
        _c["chunking_strategy"] = "definition_split"
        _new.append(_c)
    RAW[_dataset_name] = _records + _new
    print(f"{_dataset_name}: +{len(_defs)} تعريفًا → {len(RAW[_dataset_name])}")


ar_article: +19 تعريفًا → 215
en_article: +23 تعريفًا → 275


In [9]:
# توحيد المقاطع مع الاحتفاظ بالنص المتاح، ونص البحث، والبيانات الوصفية
chunk_rows = [
 {
 "dataset": dataset_name,
 "row_index": row_index,
 "chunk_id": record.get("chunk_id"),
 "document_id": record.get("document_id"),
 "language": record.get("language"),
 "article_number": record.get("article_number"),
 "covered_article_numbers": record.get("covered_article_numbers"),
 "article_title": record.get("article_title"),
 "document_title": record.get("document_title"),
 "chunking_strategy": record.get("chunking_strategy", record.get("chunking")),
 "display_text": (
 record.get("text")
 if record.get("text") is not None
 else record.get("clean_text", "")
 ),
 "display_text_field": (
 "text" if record.get("text") is not None else "clean_text"
 ),
 "search_text": (
 record.get("search_text")
 if record.get("search_text")
 else (
 record.get("text")
 if record.get("text") is not None
 else record.get("clean_text", "")
 )
 ),
 "source_url": record.get("source_url"),
 "references": record.get("references"),
 "raw_record": record,
 }
 for dataset_name in ("ar_fixed", "ar_article", "en_fixed", "en_article")
 for row_index, record in enumerate(RAW[dataset_name])
]

chunks_df = pd.DataFrame(chunk_rows)

def missing_value_count(series):
 return int(
 (series.isna() | series.astype(str).str.strip().eq("")).sum()
 )

integrity_rows = [
 {
 "dataset": dataset_name,
 "records": len(group),
 "missing_chunk_id": missing_value_count(group["chunk_id"]),
 "duplicate_chunk_id_rows": int(
 group.loc[
 group["chunk_id"].notna()
 & group["chunk_id"].astype(str).str.strip().ne(""),
 "chunk_id"
 ].duplicated(keep=False).sum()
 ),
 "missing_document_id": missing_value_count(group["document_id"]),
 "missing_article_number": missing_value_count(group["article_number"]),
 "missing_covered_article_numbers": missing_value_count(
 group["covered_article_numbers"]
 ),
 "missing_display_text": missing_value_count(group["display_text"]),
 "missing_search_text": missing_value_count(group["search_text"]),
 "missing_source_url": missing_value_count(group["source_url"]),
 "mean_display_text_chars": group["display_text"].astype(str).str.len().mean(),
 "min_display_text_chars": group["display_text"].astype(str).str.len().min(),
 "max_display_text_chars": group["display_text"].astype(str).str.len().max(),
 }
 for dataset_name, group in chunks_df.groupby("dataset", sort=False)
]

integrity_df = pd.DataFrame(integrity_rows)

print("أعداد السجلات في الجدول الموحد:")
display(chunks_df.groupby(["dataset", "language"]).size().rename("records").reset_index())

print("تقرير سلامة المقاطع:")
display(integrity_df)


أعداد السجلات في الجدول الموحد:


,dataset,language,records
0,ar_article,ar,215
1,ar_fixed,ar,241
2,en_article,en,275
3,en_fixed,en,181


تقرير سلامة المقاطع:


,dataset,records,missing_chunk_id,duplicate_chunk_id_rows,missing_document_id,missing_article_number,missing_covered_article_numbers,missing_display_text,missing_search_text,missing_source_url,mean_display_text_chars,min_display_text_chars,max_display_text_chars
0,ar_fixed,241,0,0,0,69,241,0,0,0,655.842324,7,1319
1,ar_article,215,0,0,0,53,215,0,0,0,696.967442,7,2100
2,en_fixed,181,0,0,0,181,0,0,0,0,1166.513812,160,1199
3,en_article,275,0,0,0,0,275,0,0,0,686.960000,9,1199


In [10]:
import re
from rank_bm25 import BM25Okapi

# تقسيم نص البحث إلى كلمات عربية أو لاتينية أو أرقام.
# النص القانوني الأصلي لا يتغير بهذه الخطوة.
TOKEN_PATTERN = re.compile(r"[\u0600-\u06FF]+|[a-zA-Z0-9]+", re.UNICODE)

def tokenize(text):
 return TOKEN_PATTERN.findall(str(text or "").lower())

BM25_INDEXES = {}
bm25_build_report = []

for dataset_name, group in chunks_df.groupby("dataset", sort=False):
 group = group.reset_index(drop=True)
 tokenized_documents = [
 tokenize(text)
 for text in group["search_text"]
 ]

 empty_token_documents = sum(
 len(tokens) == 0
 for tokens in tokenized_documents
 )

 BM25_INDEXES[dataset_name] = {
 "index": BM25Okapi(tokenized_documents),
 "frame": group,
 "tokens": tokenized_documents,
 }

 bm25_build_report.append({
 "dataset": dataset_name,
 "indexed_records": len(group),
 "empty_token_documents": empty_token_documents,
 "average_tokens_per_document": (
 sum(map(len, tokenized_documents)) / len(tokenized_documents)
 if tokenized_documents else 0
 ),
 })

bm25_report = pd.DataFrame(bm25_build_report)
display(bm25_report)

assert sum(row["indexed_records"] for row in bm25_build_report) == len(chunks_df), (
 "عدد المقاطع المفهرسة لا يطابق عدد المقاطع في الجدول الأصلي."
)

print("تم بناء فهرس BM25 مستقل لكل مجموعة دون حذف السجلات.")


,dataset,indexed_records,empty_token_documents,average_tokens_per_document
0,ar_fixed,241,0,107.294606
1,ar_article,215,0,113.860465
2,en_fixed,181,0,180.812155
3,en_article,275,0,106.625455


تم بناء فهرس BM25 مستقل لكل مجموعة دون حذف السجلات.


In [11]:
import pandas as pd

# البحث في فهرس BM25 وإرجاع النتائج مرتبة مع النص الأصلي
def bm25_search(question, dataset_name, top_k=5):
 assert dataset_name in BM25_INDEXES, (
 f"المجموعة غير معروفة: {dataset_name}. "
 f"المتاح: {list(BM25_INDEXES.keys())}"
 )
 assert isinstance(top_k, int) and top_k > 0, (
 "top_k يجب أن يكون عددًا صحيحًا موجبًا."
 )

 index_data = BM25_INDEXES[dataset_name]
 frame = index_data["frame"]
 scores = index_data["index"].get_scores(tokenize(question))
 ranked_indices = scores.argsort()[::-1][:min(top_k, len(frame))]

 return [
 {
 "rank": rank,
 "chunk_id": frame.iloc[int(index)]["chunk_id"],
 "document_id": frame.iloc[int(index)]["document_id"],
 "article_number": frame.iloc[int(index)]["article_number"],
 "covered_article_numbers": frame.iloc[int(index)]["covered_article_numbers"],
 "language": frame.iloc[int(index)]["language"],
 "score": float(scores[index]),
 "text": frame.iloc[int(index)]["display_text"],
 "source_url": frame.iloc[int(index)]["source_url"],
 "dataset": dataset_name,
 }
 for rank, index in enumerate(ranked_indices, start=1)
 ]

# اختيار أول سؤال متاح بالعربية وأول سؤال متاح بالإنجليزية
gold_questions = [
 row for row in RAW["gold_qa"]
 if isinstance(row, dict) and row.get("question")
]

ar_example = next(
 (
 row for row in gold_questions
 if str(row.get("language", "")).lower().startswith("ar")
 ),
 None,
)

en_example = next(
 (
 row for row in gold_questions
 if str(row.get("language", "")).lower().startswith("en")
 ),
 None,
)

test_cases = [
 ("العربية", "ar_article", ar_example),
 ("الإنجليزية", "en_article", en_example),
]

# تشغيل BM25 وعرض النتائج المتاحة
test_outputs = [
 (
 language_name,
 example["question"],
 bm25_search(example["question"], dataset_name, top_k=3),
 )
 for language_name, dataset_name, example in test_cases
 if example is not None
]

for language_name, question, results in test_outputs:
 print(f"\nالسؤال باللغة {language_name}: {question}")
 display(pd.DataFrame(results))

print("عدد اختبارات اللغة المنفذة:", len(test_outputs))



السؤال باللغة العربية: ما المقصود بالبيانات الشخصية في النظام؟


,rank,chunk_id,document_id,article_number,covered_article_numbers,language,score,text,source_url,dataset
0,1,pdpl_ar_002__article_section__001,pdpl,2,None,ar,9.115348,1- يُطبق النظام على أي عملية مُعالجة لبيانات ش...,https://dgp.sdaia.gov.sa/wps/portal/pdp/knowle...,ar_article
1,2,implementing_regulation_ar_008__article_sectio...,implementing_regulation,8,None,ar,7.494500,الحق في طلب إتلاف البيانات الشخصية على جهة الت...,https://dgp.sdaia.gov.sa/wps/portal/pdp/knowle...,ar_article
2,3,pdpl_ar_031__article_section__001,pdpl,31,None,ar,7.328369,دون إخلال بما ورد في المادة (الثامنة عشرة) من ...,https://dgp.sdaia.gov.sa/wps/portal/pdp/knowle...,ar_article



السؤال باللغة الإنجليزية: What does the PDPL mean by 'Personal Data'?


,rank,chunk_id,document_id,article_number,covered_article_numbers,language,score,text,source_url,dataset
0,1,implementing_regulation_010_chunk_01,implementing_regulation,10,None,en,8.766376,The Controller is required to provide appropri...,https://dgp.sdaia.gov.sa/wps/portal/pdp/knowle...,en_article
1,2,pdpl_010_chunk_02,pdpl,10,None,en,8.382954,ed or stored in a form that makes it possible ...,https://dgp.sdaia.gov.sa/wps/portal/pdp/knowle...,en_article
2,3,data_monetization_policy_012_chunk_01,data_monetization_policy,12,None,en,8.333420,1. The government entity shall: A. Ensure that...,https://dgp.sdaia.gov.sa/wps/portal/pdp/knowle...,en_article


عدد اختبارات اللغة المنفذة: 2


In [12]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

# إعداد مجلد حفظ المخرجات
OUTPUT_DIR = Path("/kaggle/working/stage2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# إعداد نموذج التضمين متعدد اللغات
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMBEDDING_BATCH_SIZE = 16 if DEVICE == "cuda" else 4

print("النموذج:", EMBEDDING_MODEL_NAME)
print("الجهاز:", DEVICE)
print("جاري تحميل النموذج؛ قد يستغرق التنزيل الأول عدة دقائق...")

# لا يُستخدم أي بديل تلقائيًا عند فشل التحميل
EMBEDDING_MODEL = SentenceTransformer(
 EMBEDDING_MODEL_NAME,
 device=DEVICE,
)

# الحد الأقصى يشمل البادئة والتوكنات الخاصة بالنموذج
EMBEDDING_MODEL.max_seq_length = 512
EMBEDDING_MODEL.eval()

print("\nتم تحميل النموذج بنجاح.")
print("أبعاد المتجه:", EMBEDDING_MODEL.get_sentence_embedding_dimension())
print("الحد الأقصى لطول الإدخال بالتوكنات:", EMBEDDING_MODEL.max_seq_length)
print("حجم دفعة التضمين:", EMBEDDING_BATCH_SIZE)


النموذج: intfloat/multilingual-e5-base
الجهاز: cuda
جاري تحميل النموذج؛ قد يستغرق التنزيل الأول عدة دقائق...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


تم تحميل النموذج بنجاح.
أبعاد المتجه: 768
الحد الأقصى لطول الإدخال بالتوكنات: 512
حجم دفعة التضمين: 16


/tmp/ipykernel_23/1603867836.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("أبعاد المتجه:", EMBEDDING_MODEL.get_sentence_embedding_dimension())


In [13]:
import pandas as pd

# قياس أطوال نصوص المقاطع باستخدام tokenizer الخاص بـE5
tokenizer = EMBEDDING_MODEL.tokenizer
MAX_TOKENS = EMBEDDING_MODEL.max_seq_length

token_audit_rows = []

for row in chunks_df.itertuples(index=False):
 passage = "passage: " + str(row.display_text)

 # لا نقصّ النص هنا؛ نريد قياس طوله الكامل قبل التضمين
 encoded = tokenizer(
 passage,
 add_special_tokens=True,
 truncation=False,
 )
 token_count = len(encoded["input_ids"])

 token_audit_rows.append({
 "dataset": row.dataset,
 "chunk_id": row.chunk_id,
 "token_count": token_count,
 "exceeds_model_limit": token_count > MAX_TOKENS,
 })

token_audit_df = pd.DataFrame(token_audit_rows)

token_summary = (
 token_audit_df
 .groupby("dataset")
 .agg(
 chunks=("chunk_id", "count"),
 mean_tokens=("token_count", "mean"),
 max_tokens=("token_count", "max"),
 over_512=("exceeds_model_limit", "sum"),
 )
 .reset_index()
)

token_summary["mean_tokens"] = token_summary["mean_tokens"].round(1)
token_summary["percent_over_limit"] = (
 token_summary["over_512"] / token_summary["chunks"] * 100
).round(2)

print(f"حد النموذج: {MAX_TOKENS} token")
display(token_summary)

# عرض المقاطع التي قد تُقتطع، إن وُجدت
long_chunks = (
 token_audit_df[token_audit_df["exceeds_model_limit"]]
 .sort_values("token_count", ascending=False)
 .head(10)
)

print("أطول المقاطع التي تتجاوز الحد — إن وُجدت:")
display(long_chunks)


حد النموذج: 512 token


,dataset,chunks,mean_tokens,max_tokens,over_512,percent_over_limit
0,ar_article,215,192.5,504,0,0.0
1,ar_fixed,241,181.1,304,0,0.0
2,en_article,275,148.3,285,0,0.0
3,en_fixed,181,248.4,294,0,0.0


أطول المقاطع التي تتجاوز الحد — إن وُجدت:


,dataset,chunk_id,token_count,exceeds_model_limit


In [14]:
import numpy as np
from pathlib import Path

# إنشاء مجلد حفظ المتجهات
OUTPUT_DIR = Path("/kaggle/working/stage2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# بناء وحفظ embeddings مستقلة لكل مجموعة
DENSE_INDEXES = {}
embedding_report = []

for name, group in chunks_df.groupby("dataset", sort=False):
 texts = group["display_text"].astype(str).tolist()

 # تضمين المقاطع ببادئة passage وتطبيع المتجهات
 vectors = EMBEDDING_MODEL.encode(
 ["passage: " + text for text in texts],
 batch_size=EMBEDDING_BATCH_SIZE,
 show_progress_bar=True,
 convert_to_numpy=True,
 normalize_embeddings=True,
 ).astype("float32")

 # فحص عدد المتجهات والتأكد من تطبيعها
 assert vectors.shape[0] == len(group)
 assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-3)

 # حفظ المتجهات في ملف مستقل لكل مجموعة
 np.save(OUTPUT_DIR / f"dense_embeddings_{name}.npy", vectors)
 DENSE_INDEXES[name] = vectors

 embedding_report.append({
 "dataset": name,
 "عدد المقاطع": len(group),
 "أبعاد المتجه": vectors.shape[1],
 "ملف الحفظ": f"dense_embeddings_{name}.npy",
 })

# عرض ملخص الفهارس
display(pd.DataFrame(embedding_report))


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

,dataset,عدد المقاطع,أبعاد المتجه,ملف الحفظ
0,ar_fixed,241,768,dense_embeddings_ar_fixed.npy
1,ar_article,215,768,dense_embeddings_ar_article.npy
2,en_fixed,181,768,dense_embeddings_en_fixed.npy
3,en_article,275,768,dense_embeddings_en_article.npy


### بناء فهارس FAISS

ننشئ فهرسًا مستقلًا لكل مجموعة. نستخدم `IndexFlatIP`؛ وبما أن متجهات E5 مطبّعة، فإن البحث بالضرب الداخلي يعادل حساب cosine similarity.


In [15]:
# تعريف أسماء المجموعات من الفهارس الموجودة
DATASET_NAMES = list(DENSE_INDEXES.keys())
print(DATASET_NAMES)


['ar_fixed', 'ar_article', 'en_fixed', 'en_article']


In [16]:
# بناء وحفظ فهرس FAISS لكل مجموعة
import faiss
import numpy as np

names = list(DENSE_INDEXES.keys())

# إنشاء فهارس Inner Product؛ المتجهات مطبّعة لذا تعادل cosine
FAISS_INDEXES = {
 name: faiss.IndexFlatIP(DENSE_INDEXES[name].shape[1])
 for name in names
}

# إضافة embeddings إلى الفهارس
[
 FAISS_INDEXES[name].add(
 np.ascontiguousarray(DENSE_INDEXES[name], dtype="float32")
 )
 for name in names
]

# التحقق من أعداد المتجهات
assert all(
 FAISS_INDEXES[name].ntotal
 == int(chunks_df["dataset"].eq(name).sum())
 for name in names
)

# حفظ ملفات الفهارس
[
 faiss.write_index(
 FAISS_INDEXES[name],
 str(OUTPUT_DIR / f"faiss_{name}.index")
 )
 for name in names
]

print({
 name: FAISS_INDEXES[name].ntotal
 for name in names
})


{'ar_fixed': 241, 'ar_article': 215, 'en_fixed': 181, 'en_article': 275}


In [17]:
# تجهيز صفوف المقاطع بالترتيب نفسه المستخدم عند بناء embeddings
DENSE_FRAMES = {
 name: group.reset_index(drop=True)
 for name, group in chunks_df.groupby("dataset", sort=False)
}

# البحث الدلالي باستخدام FAISS بدل البحث المباشر عبر NumPy
def dense_search(question, dataset_name, top_k=5):
 assert dataset_name in FAISS_INDEXES, f"مجموعة غير معروفة: {dataset_name}"
 assert isinstance(top_k, int) and top_k > 0, "top_k يجب أن يكون موجبًا."

 # تضمين السؤال بالبادئة المطلوبة لـE5
 query_vector = EMBEDDING_MODEL.encode(
 ["query: " + str(question)],
 convert_to_numpy=True,
 normalize_embeddings=True,
 ).astype("float32")

 # البحث في فهرس المجموعة المحددة
 k = min(top_k, FAISS_INDEXES[dataset_name].ntotal)
 scores, indices = FAISS_INDEXES[dataset_name].search(query_vector, k)
 frame = DENSE_FRAMES[dataset_name]

 # إرجاع بيانات المقطع ونصه الأصلي المتاح
 return [
 {
 "rank": rank,
 "chunk_id": frame.iloc[int(index)]["chunk_id"],
 "document_id": frame.iloc[int(index)]["document_id"],
 "article_number": frame.iloc[int(index)]["article_number"],
 "covered_article_numbers": frame.iloc[int(index)]["covered_article_numbers"],
 "language": frame.iloc[int(index)]["language"],
 "score": float(score),
 "text": frame.iloc[int(index)]["display_text"],
 "source_url": frame.iloc[int(index)]["source_url"],
 "dataset": dataset_name,
 }
 for rank, (score, index) in enumerate(zip(scores[0], indices[0]), start=1)
 if index >= 0
 ]

# اختبار سريع على سؤال عربي وآخر إنجليزي
dense_test_rows = [
 {
 "language": "العربية",
 "question": "ما المقصود بالبيانات الشخصية في النظام؟",
 "dataset": "ar_article",
 },
 {
 "language": "الإنجليزية",
 "question": "What does the PDPL mean by 'Personal Data'?",
 "dataset": "en_article",
 },
]

for example in dense_test_rows:
 results = dense_search(
 example["question"], example["dataset"], top_k=3
 )
 print(f"\nDense عبر FAISS — {example['language']}")
 print("السؤال:", example["question"])
 display(pd.DataFrame(results)[
 ["rank", "chunk_id", "document_id", "article_number", "score", "text"]
 ])



Dense عبر FAISS — العربية
السؤال: ما المقصود بالبيانات الشخصية في النظام؟


,rank,chunk_id,document_id,article_number,score,text
0,1,pdpl_ar_004__article_section__001,pdpl,4,0.859816,يكون لصاحب البيانات الشخصية -وفقاً للأحكام الو...
1,2,pdpl_ar_002__article_section__001,pdpl,2,0.856464,1- يُطبق النظام على أي عملية مُعالجة لبيانات ش...
2,3,pdpl_ar_031__article_section__001,pdpl,31,0.856315,دون إخلال بما ورد في المادة (الثامنة عشرة) من ...



Dense عبر FAISS — الإنجليزية
السؤال: What does the PDPL mean by 'Personal Data'?


,rank,chunk_id,document_id,article_number,score,text
0,1,pdpl_010_chunk_01,pdpl,10,0.836771,The Controller may only collect Personal Data ...
1,2,pdpl_001_chunk_01,pdpl,1,0.834124,"For the purpose of implementing this Law, the ..."
2,3,pdpl_001_chunk_02,pdpl,1,0.833213,"linking, blocking, erasing and destroying data..."


In [18]:
# مسح نتائج البحث المخزنة لإجبار استخدام FAISS
RETRIEVAL_CACHE.clear() if "RETRIEVAL_CACHE" in globals() else None


In [19]:
# البحث عن المقاطع الأقرب إلى السؤال باستخدام E5
def dense_search(question, dataset_name, top_k=5):
 assert dataset_name in DENSE_INDEXES, f"مجموعة غير معروفة: {dataset_name}"
 assert isinstance(top_k, int) and top_k > 0, "top_k يجب أن يكون عددًا موجبًا."

 frame = BM25_INDEXES[dataset_name]["frame"]

 # تضمين السؤال ببادئة query ومقارنته بمتجهات المقاطع المطبّعة
 query_vector = EMBEDDING_MODEL.encode(
 ["query: " + str(question)],
 convert_to_numpy=True,
 normalize_embeddings=True,
 )[0].astype("float32")

 scores = DENSE_INDEXES[dataset_name] @ query_vector
 order = np.argsort(-scores)[:min(top_k, len(frame))]

 return [
 {
 "rank": rank,
 "chunk_id": frame.iloc[int(i)]["chunk_id"],
 "document_id": frame.iloc[int(i)]["document_id"],
 "article_number": frame.iloc[int(i)]["article_number"],
 "language": frame.iloc[int(i)]["language"],
 "score": float(scores[i]),
 "text": frame.iloc[int(i)]["display_text"],
 "source_url": frame.iloc[int(i)]["source_url"],
 "dataset": dataset_name,
 }
 for rank, i in enumerate(order, start=1)
 ]

# تجربة Dense على السؤالين العربي والإنجليزي المستخدمين مع BM25
for language_name, question, bm25_results in test_outputs:
 dataset_name = "ar_article" if language_name == "العربية" else "en_article"
 dense_results = dense_search(question, dataset_name, top_k=3)

 print(f"\nDense Retrieval — {language_name}")
 print("السؤال:", question)

 display(pd.DataFrame(dense_results)[
 ["rank", "chunk_id", "document_id", "article_number", "score", "text"]
 ])



Dense Retrieval — العربية
السؤال: ما المقصود بالبيانات الشخصية في النظام؟


,rank,chunk_id,document_id,article_number,score,text
0,1,pdpl_ar_004__article_section__001,pdpl,4,0.859816,يكون لصاحب البيانات الشخصية -وفقاً للأحكام الو...
1,2,pdpl_ar_002__article_section__001,pdpl,2,0.856464,1- يُطبق النظام على أي عملية مُعالجة لبيانات ش...
2,3,pdpl_ar_031__article_section__001,pdpl,31,0.856315,دون إخلال بما ورد في المادة (الثامنة عشرة) من ...



Dense Retrieval — الإنجليزية
السؤال: What does the PDPL mean by 'Personal Data'?


,rank,chunk_id,document_id,article_number,score,text
0,1,pdpl_010_chunk_01,pdpl,10,0.836771,The Controller may only collect Personal Data ...
1,2,pdpl_001_chunk_01,pdpl,1,0.834124,"For the purpose of implementing this Law, the ..."
2,3,pdpl_001_chunk_02,pdpl,1,0.833213,"linking, blocking, erasing and destroying data..."


In [20]:
# دمج ترتيبي BM25 وDense باستخدام Reciprocal Rank Fusion
import pandas as pd

RRF_K = 60
RETRIEVAL_CANDIDATES = 50

def hybrid_search(question, dataset_name, top_k=5):
 assert dataset_name in DENSE_INDEXES, f"اسم المجموعة غير صحيح: {dataset_name}"
 assert isinstance(top_k, int) and top_k > 0, "top_k يجب أن يكون عددًا موجبًا."

 # جلب المرشحين من الطريقتين داخل المجموعة نفسها
 candidate_k = min(
 max(top_k, RETRIEVAL_CANDIDATES),
len(BM25_INDEXES[dataset_name]["frame"])
 )
 bm25_results = bm25_search(question, dataset_name, candidate_k)
 dense_results = dense_search(question, dataset_name, candidate_k)

 # إضافة اسم مصدر الترتيب لكل نتيجة
 rows = (
 [dict(result, source="bm25") for result in bm25_results]
 + [dict(result, source="dense") for result in dense_results]
 )
 rank_df = pd.DataFrame(rows)

 # حساب مساهمة RRF لكل رتبة
 rank_df["rrf_component"] = 1 / (RRF_K + rank_df["rank"])

 # جمع مساهمات RRF لكل مقطع
 scores = (
 rank_df.groupby("chunk_id")["rrf_component"]
 .sum()
 .rename("rrf_score")
 )

 # استخراج رتبة كل مقطع في BM25 وفي Dense
 ranks = rank_df.pivot_table(
 index="chunk_id",
 columns="source",
 values="rank",
 aggfunc="min"
 ).rename(columns={
 "bm25": "bm25_rank",
 "dense": "dense_rank"
 })

 # الاحتفاظ ببيانات المقطع والنص الأصلي ثم إضافة الرتب والدرجة
 metadata = (
 rank_df.drop_duplicates("chunk_id")
 .set_index("chunk_id")
 .drop(columns=["source", "rrf_component", "rank", "score"], errors="ignore")
 )
 combined = metadata.join(ranks).join(scores)
 combined = combined.sort_values("rrf_score", ascending=False).head(top_k)
 combined = combined.reset_index()

 # تعيين الرتبة النهائية وإرجاع النتائج
 combined.insert(0, "rank", range(1, len(combined) + 1))
 combined["score"] = combined["rrf_score"]

 return combined.to_dict(orient="records")

# اختبار RRF على السؤال العربي والإنجليزي السابقين
for language_name, question, _ in test_outputs:
 dataset_name = "ar_article" if language_name == "العربية" else "en_article"
 results = hybrid_search(question, dataset_name, top_k=5)

 print(f"\nHybrid RRF — {language_name}")
 print("السؤال:", question)

 display(pd.DataFrame(results)[[
 "rank",
 "chunk_id",
 "document_id",
 "article_number",
 "bm25_rank",
 "dense_rank",
 "rrf_score"
 ]])



Hybrid RRF — العربية
السؤال: ما المقصود بالبيانات الشخصية في النظام؟


,rank,chunk_id,document_id,article_number,bm25_rank,dense_rank,rrf_score
0,1,pdpl_ar_002__article_section__001,pdpl,2,1.0,2.0,0.032522
1,2,pdpl_ar_031__article_section__001,pdpl,31,3.0,3.0,0.031746
2,3,appointing_dpo_ar_001__article_section__001,appointing_dpo,1,4.0,4.0,0.031250
3,4,implementing_regulation_ar_015__article_sectio...,implementing_regulation,15,13.0,7.0,0.028624
4,5,implementing_regulation_ar_008__article_sectio...,implementing_regulation,8,2.0,21.0,0.028475



Hybrid RRF — الإنجليزية
السؤال: What does the PDPL mean by 'Personal Data'?


,rank,chunk_id,document_id,article_number,bm25_rank,dense_rank,rrf_score
0,1,implementing_regulation_012_chunk_01,implementing_regulation,12,6.0,9.0,0.029644
1,2,pdpl_010_chunk_02,pdpl,10,2.0,16.0,0.029287
2,3,implementing_regulation_001_chunk_02,implementing_regulation,1,21.0,4.0,0.027971
3,4,pdpl_015_chunk_01,pdpl,15,10.0,19.0,0.026944
4,5,implementing_regulation_004_chunk_03,implementing_regulation,4,14.0,15.0,0.026847


In [21]:
# تحميل Cross-Encoder لإعادة ترتيب المقاطع المسترجعة
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

print("جاري تحميل:", RERANKER_MODEL_NAME)

RERANKER = CrossEncoder(
 RERANKER_MODEL_NAME,
 device=DEVICE,
 max_length=512,
 activation_fn=torch.nn.Identity(),
)

print("تم تحميل نموذج إعادة الترتيب بنجاح.")
print("الجهاز:", DEVICE)


جاري تحميل: BAAI/bge-reranker-v2-m3


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

تم تحميل نموذج إعادة الترتيب بنجاح.
الجهاز: cuda


In [22]:
# إعادة ترتيب مرشحي RRF باستخدام Cross-Encoder
def rerank_search(question, dataset_name, top_k=5, candidate_k=20):
 candidates = hybrid_search(question, dataset_name, top_k=candidate_k)

 # حساب درجات أزواج السؤال والمقاطع
 pairs = [[question, item["text"]] for item in candidates]
 scores = np.asarray(RERANKER.predict(pairs), dtype=float)
 order = np.argsort(-scores)[:top_k]

 # إرجاع النتائج مرتبة مع حفظ درجتي RRF وإعادة الترتيب
 return [
 {
 **candidates[int(i)],
 "rank": rank,
 "rrf_score": candidates[int(i)]["score"],
 "reranker_score": float(scores[int(i)]),
 }
 for rank, i in enumerate(order, start=1)
 ]

# اختبار إعادة الترتيب على السؤالين السابقين
rerank_test_outputs = [
 (
 language_name,
 question,
 rerank_search(
 question,
 "ar_article" if language_name == "العربية" else "en_article",
 top_k=5,
 candidate_k=20,
 ),
 )
 for language_name, question, _ in test_outputs
]

# تجهيز جدول عرض موحد للاختبارين
rerank_display_rows = [
 {
 "اللغة": language_name,
 "السؤال": question,
 "الترتيب": result["rank"],
 "معرّف المقطع": result["chunk_id"],
 "المستند": result["document_id"],
 "المادة": result["article_number"],
 "درجة RRF": result["rrf_score"],
 "درجة Reranker": result["reranker_score"],
 "النص": result["text"],
 }
 for language_name, question, results in rerank_test_outputs
 for result in results
]

rerank_display = pd.DataFrame(rerank_display_rows)
rerank_display[["درجة RRF", "درجة Reranker"]] = (
 rerank_display[["درجة RRF", "درجة Reranker"]].round(4)
)

display(rerank_display)


,اللغة,السؤال,الترتيب,معرّف المقطع,المستند,المادة,درجة RRF,درجة Reranker,النص
0,العربية,ما المقصود بالبيانات الشخصية في النظام؟,1,pdpl_ar_002__article_section__001,pdpl,2,0.0325,2.2920,1- يُطبق النظام على أي عملية مُعالجة لبيانات ش...
1,العربية,ما المقصود بالبيانات الشخصية في النظام؟,2,implementing_regulation_ar_002__article_sectio...,implementing_regulation,2,0.0223,1.9804,الاستخدام الشخصي أو العائلي لا تسري أحكام النظ...
2,العربية,ما المقصود بالبيانات الشخصية في النظام؟,3,pdpl_ar_013__article_section__001,pdpl,13,0.0281,0.9893,على جهة التحكم، في حالة جمع البيانات الشخصية م...
3,العربية,ما المقصود بالبيانات الشخصية في النظام؟,4,implementing_regulation_ar_020__article_sectio...,implementing_regulation,20,0.0230,0.6713,الإفصاح عن البيانات الشخصية يشترط في الإفصاح ع...
4,العربية,ما المقصود بالبيانات الشخصية في النظام؟,5,pdpl_ar_004__article_section__001,pdpl,4,0.0283,0.6347,يكون لصاحب البيانات الشخصية -وفقاً للأحكام الو...
5,الإنجليزية,What does the PDPL mean by 'Personal Data'?,1,pdpl_001_chunk_02,pdpl,1,0.0264,-3.0860,"linking, blocking, erasing and destroying data..."
6,الإنجليزية,What does the PDPL mean by 'Personal Data'?,2,implementing_regulation_004_chunk_03,implementing_regulation,4,0.0268,-4.1354,"ial requirements, or to achieve a Public Inter..."
7,الإنجليزية,What does the PDPL mean by 'Personal Data'?,3,pdpl_013_chunk_01,pdpl,13,0.0239,-4.2938,When collecting Personal Data directly from th...
8,الإنجليزية,What does the PDPL mean by 'Personal Data'?,4,pdpl_010_chunk_01,pdpl,10,0.0164,-4.3795,The Controller may only collect Personal Data ...
9,الإنجليزية,What does the PDPL mean by 'Personal Data'?,5,pdpl_010_chunk_02,pdpl,10,0.0293,-4.6743,ed or stored in a form that makes it possible ...


In [23]:
# التحقق من أن dense_search الحالية تستدعي فهرس FAISS
import inspect

print(inspect.getsource(dense_search))


def dense_search(question, dataset_name, top_k=5):
 assert dataset_name in DENSE_INDEXES, f"مجموعة غير معروفة: {dataset_name}"
 assert isinstance(top_k, int) and top_k > 0, "top_k يجب أن يكون عددًا موجبًا."

 frame = BM25_INDEXES[dataset_name]["frame"]

 # تضمين السؤال ببادئة query ومقارنته بمتجهات المقاطع المطبّعة
 query_vector = EMBEDDING_MODEL.encode(
 ["query: " + str(question)],
 convert_to_numpy=True,
 normalize_embeddings=True,
 )[0].astype("float32")

 scores = DENSE_INDEXES[dataset_name] @ query_vector
 order = np.argsort(-scores)[:min(top_k, len(frame))]

 return [
 {
 "rank": rank,
 "chunk_id": frame.iloc[int(i)]["chunk_id"],
 "document_id": frame.iloc[int(i)]["document_id"],
 "article_number": frame.iloc[int(i)]["article_number"],
 "language": frame.iloc[int(i)]["language"],
 "score": float(scores[i]),
 "text": frame.iloc[int(i)]["display_text"],
 "source_url": frame.iloc[int(i)]["source_url"],
 "dataset": dataset_name,
 }
 for rank, i in enumerate(order, start=1)
 ]



In [24]:
# تحميل Gold QA مع الاحتفاظ بالحقول المتداخلة
gold_df = pd.json_normalize(RAW["gold_qa"])

# تلخيص التوزيعات الفعلية للحقول المهمة
status_columns = [
 "language",
 "answerable",
 "in_corpus",
 "eligible_for_article_retrieval_eval",
 "gold_reference_in_chunk_corpus",
 "evaluation_status",
 "evidence_review_status",
]

status_summary = (
 gold_df[status_columns]
 .fillna("مفقود")
 .astype(str)
 .melt(var_name="الحقل", value_name="القيمة")
 .groupby(["الحقل", "القيمة"], dropna=False)
 .size()
 .rename("عدد الأسئلة")
 .reset_index()
)

print("عدد أسئلة Gold:", len(gold_df))
display(status_summary)

# معاينة أسئلة ومراجعها الفعلية
review_columns = [
 "question_id",
 "language",
 "question",
 "answerable",
 "in_corpus",
 "eligible_for_article_retrieval_eval",
 "gold_reference_in_chunk_corpus",
 "gold_document_id",
 "gold_article_number",
 "gold_references",
 "best_chunk_id",
 "evaluation_status",
]

display(gold_df[review_columns].head(12))


عدد أسئلة Gold: 90


,الحقل,القيمة,عدد الأسئلة
0,answerable,False,15
1,answerable,True,75
2,eligible_for_article_retrieval_eval,False,58
3,eligible_for_article_retrieval_eval,True,32
4,evaluation_status,abstention_only,10
5,evaluation_status,needs_section_mapping,17
6,evaluation_status,out_of_corpus,5
7,evaluation_status,provisional_article_match,53
8,evaluation_status,unanswerable,5
9,evidence_review_status,full_evidence_text_match,53


,question_id,language,question,answerable,in_corpus,eligible_for_article_retrieval_eval,gold_reference_in_chunk_corpus,gold_document_id,gold_article_number,gold_references,best_chunk_id,evaluation_status
0,AR001,ar,ما المقصود بالبيانات الشخصية في النظام؟,True,True,True,Yes,pdpl,1.0,"[{'document_id': 'pdpl', 'article_number': 1}]",pdpl_ar_001_chunk_01,provisional_article_match
1,AR002,ar,ما المقصود بمعالجة البيانات الشخصية؟,True,True,True,Yes,pdpl,1.0,"[{'document_id': 'pdpl', 'article_number': 1}]",pdpl_ar_001_chunk_01,provisional_article_match
2,AR003,ar,ما أمثلة البيانات الشخصية الحساسة؟,True,True,True,Yes,pdpl,1.0,"[{'document_id': 'pdpl', 'article_number': 1}]",pdpl_ar_001_chunk_02,provisional_article_match
3,AR004,ar,ما الفرق بين جهة التحكم وجهة المعالجة؟,True,True,True,Yes,pdpl,1.0,"[{'document_id': 'pdpl', 'article_number': 1}]",pdpl_ar_001_chunk_03,provisional_article_match
4,AR005,ar,هل الاستخدام الشخصي أو العائلي للبيانات مستثنى...,True,True,True,Yes,pdpl,2.0,"[{'document_id': 'pdpl', 'article_number': 2}]",pdpl_ar_002_chunk_01,provisional_article_match
5,AR006,ar,ما الحقوق الأساسية لصاحب البيانات الشخصية؟,True,True,True,Yes,pdpl,4.0,"[{'document_id': 'pdpl', 'article_number': 4}]",pdpl_ar_004_chunk_01,provisional_article_match
6,AR007,ar,هل يحق لصاحب البيانات الرجوع عن موافقته؟,True,True,True,Yes,pdpl,5.0,"[{'document_id': 'pdpl', 'article_number': 5}]",pdpl_ar_005_chunk_01,provisional_article_match
7,AR008,ar,هل يمكن معالجة بيانات غير حساسة لمصلحة مشروعة ...,True,True,True,Yes,pdpl,6.0,"[{'document_id': 'pdpl', 'article_number': 6}]",pdpl_ar_006_chunk_01,provisional_article_match
8,AR009,ar,ما الذي يجب تضمينه في سياسة الخصوصية؟,True,True,True,Yes,pdpl,12.0,"[{'document_id': 'pdpl', 'article_number': 12}]",pdpl_ar_012_chunk_01,provisional_article_match
9,AR010,ar,متى يجب إتلاف البيانات الشخصية؟,True,True,True,Yes,pdpl,18.0,"[{'document_id': 'pdpl', 'article_number': 18}]",pdpl_ar_018_chunk_01,provisional_article_match


In [25]:
# فحص تقاطع الأهلية والحالة واللغة قبل تحديد أسئلة التقييم
cross_tab = pd.crosstab(
 [
 gold_df["language"],
 gold_df["eligible_for_article_retrieval_eval"],
 gold_df["evaluation_status"],
 ],
 gold_df["gold_reference_in_chunk_corpus"].fillna("مفقود"),
 margins=True
)

display(cross_tab)

# فحص حالة اعتماد السؤال للتقييم النهائي إن كان الحقل موجودًا
approved_summary = gold_df["approved_for_final_eval"].fillna("مفقود").value_counts(
 dropna=False
).rename_axis("approved_for_final_eval").reset_index(name="عدد الأسئلة")

display(approved_summary)

# مقارنة best_chunk_id بمعرفات المقاطع الفعلية
actual_chunk_ids = set(chunks_df["chunk_id"].astype(str))
gold_best_ids = gold_df["best_chunk_id"].dropna().astype(str)

best_chunk_audit = pd.DataFrame({
 "question_id": gold_df.loc[gold_df["best_chunk_id"].notna(), "question_id"].astype(str),
 "best_chunk_id": gold_best_ids.values,
 "exists_in_any_dataset": gold_best_ids.isin(actual_chunk_ids).values,
})

print("نتيجة تدقيق best_chunk_id:")
display(best_chunk_audit["exists_in_any_dataset"].value_counts().rename_axis(
 "معرف المقطع موجود"
).reset_index(name="عدد الأسئلة"))

display(best_chunk_audit.head(10))


gold_reference_in_chunk_corpus                                          No  \
language eligible_for_article_retrieval_eval evaluation_status               
ar       False                               abstention_only             0   
                                             needs_section_mapping       0   
                                             out_of_corpus               5   
         True                                provisional_article_match   0   
en       False                               needs_section_mapping       0   
                                             provisional_article_match   0   
                                             unanswerable                0   
All                                                                      5   

gold_reference_in_chunk_corpus                                          Yes  \
language eligible_for_article_retrieval_eval evaluation_status                
ar       False                               abstention_only              0   
                                             needs_section_mapping        0   
                                             out_of_corpus                0   
         True                                provisional_article_match   32   
en       False                               needs_section_mapping        4   
                                             provisional_article_match   21   
                                             unanswerable                 0   
All                                                                      57   

gold_reference_in_chunk_corpus                                          section_not_mapped  \
language eligible_for_article_retrieval_eval evaluation_status                               
ar       False                               abstention_only                             0   
                                             needs_section_mapping                      13   
                                             out_of_corpus                               0   
         True                                provisional_article_match                   0   
en       False                               needs_section_mapping                       0   
                                             provisional_article_match                   0   
                                             unanswerable                                0   
All                                                                                     13   

gold_reference_in_chunk_corpus                                          مفقود  \
language eligible_for_article_retrieval_eval evaluation_status                  
ar       False                               abstention_only               10   
                                             needs_section_mapping          0   
                                             out_of_corpus                  0   
         True                                provisional_article_match      0   
en       False                               needs_section_mapping          0   
                                             provisional_article_match      0   
                                             unanswerable                   5   
All                                                                        15   

gold_reference_in_chunk_corpus                                          All  
language eligible_for_article_retrieval_eval evaluation_status               
ar       False                               abstention_only             10  
                                             needs_section_mapping       13  
                                             out_of_corpus                5  
         True                                provisional_article_match   32  
en       False                               needs_section_mapping        4  
                                             provisional_article_match   21  
                                             unanswe

,approved_for_final_eval,عدد الأسئلة
0,False,90


نتيجة تدقيق best_chunk_id:


,معرف المقطع موجود,عدد الأسئلة
0,False,49
1,True,21


,question_id,best_chunk_id,exists_in_any_dataset
0,AR001,pdpl_ar_001_chunk_01,False
1,AR002,pdpl_ar_001_chunk_01,False
2,AR003,pdpl_ar_001_chunk_02,False
3,AR004,pdpl_ar_001_chunk_03,False
4,AR005,pdpl_ar_002_chunk_01,False
5,AR006,pdpl_ar_004_chunk_01,False
6,AR007,pdpl_ar_005_chunk_01,False
7,AR008,pdpl_ar_006_chunk_01,False
8,AR009,pdpl_ar_012_chunk_01,False
9,AR010,pdpl_ar_018_chunk_01,False


In [26]:
# عرض الحالات التي تحتاج ربطًا بالمادة لمراجعتها قبل التقييم
mapping_review = gold_df.loc[
 gold_df["evaluation_status"].eq("needs_section_mapping"),
 [
 "question_id", "language", "question",
 "gold_document_id", "gold_article_number",
 "gold_references", "gold_reference_original",
 "gold_reference_in_chunk_corpus",
 "evaluation_status", "evidence_review_status"
 ]
]

display(mapping_review.sort_values(["language", "question_id"]))


,question_id,language,question,gold_document_id,gold_article_number,gold_references,gold_reference_original,gold_reference_in_chunk_corpus,evaluation_status,evidence_review_status
32,AR033,ar,من هو مقدم طلب مشاركة البيانات؟,data_sharing_policy,NaN,[],التعريفات، البند 2,section_not_mapped,needs_section_mapping,full_evidence_text_match
33,AR034,ar,ما المقصود بالجهة المصدر في سياسة مشاركة البيا...,data_sharing_policy,NaN,[],التعريفات، البند 3,section_not_mapped,needs_section_mapping,full_evidence_text_match
34,AR035,ar,ما المقصود باتفاقية مشاركة البيانات؟,data_sharing_policy,NaN,[],التعريفات، البند 7,section_not_mapped,needs_section_mapping,partial_match_review
35,AR036,ar,ما المقصود بقناة التكامل الحكومية؟,data_sharing_policy,NaN,[],التعريفات، البند 12,section_not_mapped,needs_section_mapping,full_evidence_text_match
36,AR037,ar,على أي أنواع من البيانات تنطبق سياسة مشاركة ال...,data_sharing_policy,NaN,[],أولاً: النطاق، البند 1,section_not_mapped,needs_section_mapping,full_evidence_text_match
37,AR038,ar,هل تشمل سياسة مشاركة البيانات الطلبات الحكومية...,data_sharing_policy,NaN,[],أولاً: النطاق، البند 2,section_not_mapped,needs_section_mapping,full_evidence_text_match
38,AR039,ar,ما المقصود بمبدأ المرة الواحدة؟,data_sharing_policy,NaN,[],ثانياً: المبدأ الثاني,section_not_mapped,needs_section_mapping,full_evidence_text_match
39,AR040,ar,ما شرط مشروعية الغرض عند مشاركة البيانات؟,data_sharing_policy,NaN,[],ثانياً: المبدأ الثالث,section_not_mapped,needs_section_mapping,full_evidence_text_match
40,AR041,ar,ما المقصود بمنتجات البيانات؟,data_monetization_policy,NaN,[],أولاً: التعريفات، البند 3,section_not_mapped,needs_section_mapping,full_evidence_text_match
41,AR042,ar,ما المقصود بخدمات البيانات؟,data_monetization_policy,NaN,[],أولاً: التعريفات، البند 4,section_not_mapped,needs_section_mapping,full_evidence_text_match


In [27]:
# فحص مراجع الأسئلة الإنجليزية الأربعة في مجموعات المقاطع
targets = gold_df.loc[
 gold_df["language"].eq("en")
 & gold_df["evaluation_status"].eq("needs_section_mapping"),
 ["question_id", "gold_document_id", "gold_article_number", "question"]
].copy()

# تجهيز أرقام المواد الصريحة في المقاطع
direct_articles = chunks_df[
 ["dataset", "chunk_id", "document_id", "article_number"]
].rename(columns={"article_number": "article_value"})

# تجهيز أرقام المواد المغطاة في مقاطع fixed-size الإنجليزية
covered_articles = (
 chunks_df[
 ["dataset", "chunk_id", "document_id", "covered_article_numbers"]
 ]
 .explode("covered_article_numbers")
 .rename(columns={"covered_article_numbers": "article_value"})
)

# جمع نوعي مراجع المواد في جدول واحد
corpus_articles = pd.concat(
 [
 direct_articles,
 covered_articles[direct_articles.columns],
 ],
 ignore_index=True
).dropna(subset=["article_value"])

# توحيد صيغة المستند ورقم المادة قبل المطابقة الدقيقة
corpus_articles["document_key"] = (
 corpus_articles["document_id"].astype(str).str.strip().str.lower()
)
corpus_articles["article_key"] = pd.to_numeric(
 corpus_articles["article_value"], errors="coerce"
).astype("Int64").astype("string")

targets["document_key"] = (
 targets["gold_document_id"].astype(str).str.strip().str.lower()
)
targets["article_key"] = pd.to_numeric(
 targets["gold_article_number"], errors="coerce"
).astype("Int64").astype("string")

# مطابقة كل سؤال مع مقاطع المستند والمادة نفسيهما
english_mapping_audit = targets.merge(
 corpus_articles,
 on=["document_key", "article_key"],
 how="left"
)

display(
 english_mapping_audit[
 [
 "question_id",
 "gold_document_id",
 "gold_article_number",
 "dataset",
 "chunk_id",
 "question"
 ]
 ].sort_values(["question_id", "dataset", "chunk_id"])
)

# تلخيص عدد المقاطع المطابقة لكل سؤال ولكل مجموعة
display(
 english_mapping_audit
 .groupby(["question_id", "gold_document_id", "gold_article_number", "dataset"])
 ["chunk_id"]
 .nunique()
 .rename("matching_chunks")
 .reset_index()
)


,question_id,gold_document_id,gold_article_number,dataset,chunk_id,question
0,EN010,data_monetization_policy,3.0,en_article,data_monetization_policy_003_chunk_01,We are building a data product for a governmen...
1,EN010,data_monetization_policy,3.0,en_fixed,data_monetization_policy_fixed_0004,We are building a data product for a governmen...
2,EN023,data_sharing_policy,15.0,en_article,data_sharing_policy_015_chunk_01,A private company asks a government entity to ...
3,EN023,data_sharing_policy,15.0,en_article,data_sharing_policy_015_chunk_02,A private company asks a government entity to ...
4,EN023,data_sharing_policy,15.0,en_fixed,data_sharing_policy_fixed_0018,A private company asks a government entity to ...
5,EN023,data_sharing_policy,15.0,en_fixed,data_sharing_policy_fixed_0019,A private company asks a government entity to ...
6,EN023,data_sharing_policy,15.0,en_fixed,data_sharing_policy_fixed_0020,A private company asks a government entity to ...
7,EN024,secondary_data_use,2.0,en_article,secondary_data_use_002_chunk_01,What does 'Secondary Use of Data' mean in SDAI...
8,EN024,secondary_data_use,2.0,en_article,secondary_data_use_002_chunk_02,What does 'Secondary Use of Data' mean in SDAI...
9,EN024,secondary_data_use,2.0,en_fixed,secondary_data_use_fixed_0001,What does 'Secondary Use of Data' mean in SDAI...


,question_id,gold_document_id,gold_article_number,dataset,matching_chunks
0,EN010,data_monetization_policy,3.0,en_article,1
1,EN010,data_monetization_policy,3.0,en_fixed,1
2,EN023,data_sharing_policy,15.0,en_article,2
3,EN023,data_sharing_policy,15.0,en_fixed,3
4,EN024,secondary_data_use,2.0,en_article,2
5,EN024,secondary_data_use,2.0,en_fixed,3
6,EN025,hrsd_780009_en,1.0,en_fixed,12


In [28]:
# عرض السؤال EN025 والدليل الذهبي كاملًا
en025 = gold_df.loc[gold_df["question_id"].eq("EN025")].iloc[0]

print("السؤال:", en025["question"])
print("\nالدليل الذهبي:\n", en025["gold_evidence_extract"])
print("\nالمرجع:", en025["gold_reference_original"])

# تحديد مقاطع المستند في fixed-size وروابط مصادرها
hrsd_fixed = chunks_df.loc[
 chunks_df["dataset"].eq("en_fixed")
 & chunks_df["document_id"].eq(en025["gold_document_id"])
].copy()

source_urls = hrsd_fixed["source_url"].dropna().unique()

# البحث عن المستند في Article-level بالمعرف أو رابط المصدر
hrsd_article = chunks_df.loc[
 chunks_df["dataset"].eq("en_article")
 & (
 chunks_df["document_id"].astype(str).str.contains(
 r"hrsd|780009", case=False, regex=True
 )
 | chunks_df["source_url"].isin(source_urls)
 )
]

print("\nالمقاطع المحتملة في Article-level:", len(hrsd_article))
display(hrsd_article[
 ["chunk_id", "document_id", "article_number", "display_text"]
])

# معاينة مقاطع Fixed-size لتحديد المقاطع المتعلقة بحقوق الأفراد
print("\nمعاينة مقاطع Fixed-size:")
print(
 hrsd_fixed.assign(
 preview=hrsd_fixed["display_text"].str[:250]
 )[["chunk_id", "preview"]].to_string(index=False)
)


السؤال: What rights do individuals have under the HRSD Freedom of Information Policy?

الدليل الذهبي:
 First: The right to view any information that is not protected. Second: The right to know the reason for rejecting the request for viewing or obtaining the requested information. Third: The right to appeal against the decision relevant to rejecting the request to view or obtain the requested information.

المرجع: Individual Rights to View or Obtain Public Information

المقاطع المحتملة في Article-level: 0


,chunk_id,document_id,article_number,display_text



معاينة مقاطع Fixed-size:
                 chunk_id                                                                                                                                                                                                                                                    preview
hrsd_780009_en_fixed_0001 Key Principles and General Rules of Freedom of Information: Scope This Policy applies to all requests by individuals to view or obtain public information – unprotected and produced by public authorities, of whatever source, form or nature, including 
hrsd_780009_en_fixed_0002 information that, if disclosed, may result in gaining profits or avoiding losses in an illegitimate manner. 7. Scientific or technological research, or rights including intellectual property rights that, if disclosed, may result in the infringement o
hrsd_780009_en_fixed_0003 to view unprotected public information. The applicant may not need to have a certain quality or interest in this info

In [29]:
# التحقق من وجود دليل EN025 في مقاطع en_fixed
import re
import pandas as pd

en025 = gold_df.loc[gold_df["question_id"].eq("EN025")].iloc[0]
evidence = str(en025["gold_evidence_extract"])

# تطبيع للبحث الحرفي مع تجاهل اختلاف الترقيم والمسافات
normalize_evidence = lambda text: re.sub(
 r"\s+", " ",
 re.sub(r"[^a-z0-9]+", " ", str(text).lower())
).strip()

# تقسيم الدليل إلى فقراته الثلاث
evidence_parts = [
 part.strip()
 for part in re.split(r"(?i)(?=second:|third:)", evidence)
 if part.strip()
]

# اختيار مقاطع المستند وترتيبها حسب معرّف المقطع
fixed_chunks = (
 chunks_df.loc[
 chunks_df["dataset"].eq("en_fixed")
 & chunks_df["document_id"].eq("hrsd_780009_en")
 ]
 .sort_values("chunk_id")
 .copy()
)

# فحص كل فقرة من الدليل مقابل كل مقطع
chunk_texts = fixed_chunks["display_text"].astype(str).tolist()
normalized_chunk_texts = [normalize_evidence(text) for text in chunk_texts]

part_audit = pd.DataFrame([
 {
 "chunk_id": chunk_id,
 "gold_evidence_part": part,
 "exact_part_match": normalize_evidence(part) in chunk_text,
 }
 for chunk_id, chunk_text in zip(fixed_chunks["chunk_id"], normalized_chunk_texts)
 for part in evidence_parts
])

# فحص الدليل كاملًا بعد جمع مقاطع المستند
combined_document_text = normalize_evidence(" ".join(chunk_texts))
full_evidence_present = normalize_evidence(evidence) in combined_document_text

print("هل يظهر الدليل كاملًا في نص المستند المجمع؟", full_evidence_present)
print("عدد المقاطع المفحوصة:", len(fixed_chunks))
display(part_audit)


هل يظهر الدليل كاملًا في نص المستند المجمع؟ True
عدد المقاطع المفحوصة: 12


,chunk_id,gold_evidence_part,exact_part_match
0,hrsd_780009_en_fixed_0001,First: The right to view any information that ...,False
1,hrsd_780009_en_fixed_0001,Second: The right to know the reason for rejec...,False
2,hrsd_780009_en_fixed_0001,Third: The right to appeal against the decisio...,False
3,hrsd_780009_en_fixed_0002,First: The right to view any information that ...,False
4,hrsd_780009_en_fixed_0002,Second: The right to know the reason for rejec...,False
5,hrsd_780009_en_fixed_0002,Third: The right to appeal against the decisio...,False
6,hrsd_780009_en_fixed_0003,First: The right to view any information that ...,True
7,hrsd_780009_en_fixed_0003,Second: The right to know the reason for rejec...,True
8,hrsd_780009_en_fixed_0003,Third: The right to appeal against the decisio...,True
9,hrsd_780009_en_fixed_0004,First: The right to view any information that ...,False


In [30]:
import re
import unicodedata
import pandas as pd

# توحيد Unicode والمسافات فقط للمطابقة النصية الدقيقة
def normalize_evidence(text):
 text = unicodedata.normalize("NFKC", str(text or "")).casefold()
 return re.sub(r"\s+", " ", text).strip()

# اختيار الأسئلة العربية التي تحتاج ربطًا بالمقاطع
ar_questions = gold_df.loc[
 gold_df["language"].eq("ar")
 & gold_df["evaluation_status"].eq("needs_section_mapping")
]

# البحث عن الدليل كاملًا داخل مقاطع المستند نفسه
arabic_mapping_rows = [
 {
 "question_id": q["question_id"],
 "dataset": chunk["dataset"],
 "chunk_id": chunk["chunk_id"],
 "document_id": chunk["document_id"],
 "exact_evidence_match": (
 bool(normalize_evidence(q["gold_evidence_extract"]))
 and normalize_evidence(q["gold_evidence_extract"])
 in normalize_evidence(chunk["display_text"])
 ),
 }
 for _, q in ar_questions.iterrows()
 for _, chunk in chunks_df[
 chunks_df["language"].eq("ar")
 & chunks_df["document_id"].astype(str).str.casefold().eq(
 str(q["gold_document_id"]).casefold()
 )
 ].iterrows()
]

arabic_mapping_audit = pd.DataFrame(arabic_mapping_rows)

# عرض المقاطع التي طابقت مقتطف الدليل كاملًا
arabic_exact_matches = arabic_mapping_audit.loc[
 arabic_mapping_audit["exact_evidence_match"]
].sort_values(["question_id", "dataset"])

display(arabic_exact_matches)
print("عدد الأسئلة العربية ذات تطابق دليل كامل:",
 arabic_exact_matches["question_id"].nunique())


,question_id,dataset,chunk_id,document_id,exact_evidence_match
27,AR033,ar_article,data_sharing_policy_ar_001__article_section__001,data_sharing_policy,True
0,AR033,ar_fixed,data_sharing_policy_ar_001__fixed_size__001,data_sharing_policy,True
353,AR040,ar_article,data_sharing_policy_ar_005__article_section__001,data_sharing_policy,True
328,AR040,ar_fixed,data_sharing_policy_ar_005__fixed_size__001,data_sharing_policy,True
387,AR041,ar_article,data_monetization_policy_ar_002__article_secti...,data_monetization_policy,True
369,AR041,ar_fixed,data_monetization_policy_ar_002__fixed_size__001,data_monetization_policy,True
522,AR045,ar_article,data_monetization_policy_ar_004__article_secti...,data_monetization_policy,True
505,AR045,ar_fixed,data_monetization_policy_ar_004__fixed_size__001,data_monetization_policy,True


عدد الأسئلة العربية ذات تطابق دليل كامل: 4


In [31]:
import re
import unicodedata
import pandas as pd

# تطبيع عربي محافظ لتسهيل المقارنة النصية
def normalize_arabic_evidence(text):
 text = unicodedata.normalize("NFKC", str(text or "")).casefold()
 text = re.sub(r"[\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)
 text = text.replace("ـ", "")
 text = re.sub(r"[أإآٱ]", "ا", text).replace("ى", "ي")
 return re.sub(r"\s+", " ", text).strip()

# تجميع نصوص المقاطع بترتيبها داخل كل مستند ومجموعة
document_texts = (
 chunks_df.sort_values(["dataset", "document_id", "row_index"])
 .groupby(["dataset", "document_id"])["display_text"]
 .agg(lambda texts: " ".join(texts.astype(str)))
 .rename("combined_text")
 .reset_index()
)
document_texts["combined_norm"] = document_texts["combined_text"].map(
 normalize_arabic_evidence
)

# فحص مقتطف Gold كاملًا في نص المستند المجمع
arabic_document_audit = [
 {
 "question_id": q["question_id"],
 "dataset": doc["dataset"],
 "document_id": doc["document_id"],
 "evidence_found_in_combined_document": (
 bool(normalize_arabic_evidence(q["gold_evidence_extract"]))
 and normalize_arabic_evidence(q["gold_evidence_extract"])
 in doc["combined_norm"]
 ),
 }
 for _, q in ar_questions.iterrows()
 for _, doc in document_texts[
 document_texts["document_id"].astype(str).str.casefold().eq(
 str(q["gold_document_id"]).casefold()
 )
 ].iterrows()
]

arabic_document_audit = pd.DataFrame(arabic_document_audit)
display(arabic_document_audit)


,question_id,dataset,document_id,evidence_found_in_combined_document
0,AR033,ar_article,data_sharing_policy,True
1,AR033,ar_fixed,data_sharing_policy,True
2,AR033,en_article,data_sharing_policy,False
3,AR033,en_fixed,data_sharing_policy,False
4,AR034,ar_article,data_sharing_policy,True
5,AR034,ar_fixed,data_sharing_policy,True
6,AR034,en_article,data_sharing_policy,False
7,AR034,en_fixed,data_sharing_policy,False
8,AR035,ar_article,data_sharing_policy,False
9,AR035,ar_fixed,data_sharing_policy,False


In [32]:
import re
import unicodedata
import pandas as pd

# تطبيع النص للمقارنة مع تجاهل الترقيم والمسافات الزائدة
normalize_for_audit = lambda text: re.sub(
 r"\s+", " ",
 re.sub(
 r"[^\w\s\u0600-\u06FF]",
 " ",
 unicodedata.normalize("NFKC", str(text or ""))
 .casefold()
 .replace("ـ", "")
 .replace("أ", "ا")
 .replace("إ", "ا")
 .replace("آ", "ا")
 )
).strip()

# اختيار السؤالين اللذين يحتاجان مراجعة
review_questions = gold_df.loc[
 gold_df["question_id"].isin(["AR035", "AR044"]),
 [
 "question_id",
 "question",
 "gold_document_id",
 "gold_evidence_extract",
 "gold_reference_original",
 "evidence_review_status",
 ],
].copy()

# جمع المقاطع العربية لكل مستند وطريقة تقسيم
arabic_document_texts = (
 chunks_df.loc[chunks_df["language"].eq("ar")]
 .groupby(["dataset", "document_id"], as_index=False)
 .agg(
 combined_text=("display_text", lambda values: " ".join(values.astype(str))),
 document_chunk_count=("chunk_id", "size"),
 )
)

# ربط السؤال بنص مستنده في المجموعتين
review = review_questions.merge(
 arabic_document_texts,
 left_on="gold_document_id",
 right_on="document_id",
 how="left",
)

# فحص وجود الدليل كاملًا بعد التطبيع
review["normalized_full_match"] = [
 bool(normalize_for_audit(evidence))
 and normalize_for_audit(evidence) in normalize_for_audit(text)
 for evidence, text in zip(review["gold_evidence_extract"], review["combined_text"])
]

display(
 review[
 [
 "question_id",
 "dataset",
 "question",
 "gold_evidence_extract",
 "gold_reference_original",
 "evidence_review_status",
 "normalized_full_match",
 "document_chunk_count",
 ]
 ].sort_values(["question_id", "dataset"])
)


,question_id,dataset,question,gold_evidence_extract,gold_reference_original,evidence_review_status,normalized_full_match,document_chunk_count
0,AR035,ar_article,ما المقصود باتفاقية مشاركة البيانات؟,. اتفاقية مشــاركة البيانات: اتفاقية قياســية ...,التعريفات، البند 7,partial_match_review,False,19
1,AR035,ar_fixed,ما المقصود باتفاقية مشاركة البيانات؟,. اتفاقية مشــاركة البيانات: اتفاقية قياســية ...,التعريفات، البند 7,partial_match_review,False,27
2,AR044,ar_article,هل تشمل سياسة تحقيق الإيرادات البيانات المصنفة...,يستثنى من نطاق تطبيق هذه السياسة بيانات الجهات...,ثانياً: النطاق، البند 2,full_evidence_text_match,True,15
3,AR044,ar_fixed,هل تشمل سياسة تحقيق الإيرادات البيانات المصنفة...,يستثنى من نطاق تطبيق هذه السياسة بيانات الجهات...,ثانياً: النطاق، البند 2,full_evidence_text_match,True,18


In [33]:
# اختيار الأسئلة المؤكدة للمقارنة قبل حساب مقاييس الاسترجاع

# أسئلة عربية provisional المؤهلة حسب Gold QA
ar_provisional_ids = set(
 gold_df.loc[
 gold_df["language"].eq("ar")
 & gold_df["evaluation_status"].eq("provisional_article_match")
 & gold_df["eligible_for_article_retrieval_eval"].eq(True),
 "question_id"
 ].astype(str)
)

# أسئلة عربية تحقق دليلها في المستندين وطريقتي التقسيم
ar_audit_pivot = arabic_document_audit.pivot_table(
 index="question_id",
 columns="dataset",
 values="evidence_found_in_combined_document",
 aggfunc="all",
 fill_value=False
)

ar_mapped_ids = set(
 ar_audit_pivot.index[
 ar_audit_pivot["ar_fixed"] & ar_audit_pivot["ar_article"]
 ].astype(str)
)

# استبعاد AR035 لأنها لم تجتز مطابقة الدليل الكامل
ar_mapped_ids.discard("AR035")
ar_eval_ids = ar_provisional_ids | ar_mapped_ids

# أسئلة إنجليزية provisional، وثلاثة مراجع ربطناها في المجموعتين
en_provisional_ids = set(
 gold_df.loc[
 gold_df["language"].eq("en")
 & gold_df["evaluation_status"].eq("provisional_article_match")
 & gold_df["eligible_for_article_retrieval_eval"].eq(True),
 "question_id"
 ].astype(str)
)

en_mapped_ids = {"EN010", "EN023", "EN024"}
en_eval_ids = en_provisional_ids | en_mapped_ids

# إنشاء جدول أسئلة المقارنة مع إبقاء السجلات الأصلية في Gold
eval_cohort = gold_df.loc[
 gold_df["question_id"].astype(str).isin(ar_eval_ids | en_eval_ids)
].copy()

eval_cohort["evaluation_group"] = eval_cohort["language"].map({
 "ar": "primary_ar_comparison",
 "en": "primary_en_comparison",
})

# إضافة EN025 كحالة منفصلة لـen_fixed فقط، لا للمقارنة بين التقسيمين
en025_extra = gold_df.loc[gold_df["question_id"].astype(str).eq("EN025")].copy()
en025_extra["evaluation_group"] = "fixed_only_extra"

print("عدد الأسئلة العربية للمقارنة:", len(ar_eval_ids))
print("عدد الأسئلة الإنجليزية للمقارنة المشتركة:", len(en_eval_ids))
print("عدد أسئلة المقارنة الكلي:", len(eval_cohort))
print("EN025 محفوظ كتقييم إضافي لـen_fixed:", len(en025_extra))

display(
 eval_cohort.groupby(["language", "evaluation_group"])
 .size()
 .rename("عدد الأسئلة")
 .reset_index()
)

# عرض الحالات المستبعدة أو المنفصلة بوضوح
display(
 gold_df.loc[
 gold_df["question_id"].astype(str).isin(["AR035", "EN025"]),
 [
 "question_id", "language", "question",
 "evaluation_status", "gold_reference_in_chunk_corpus"
 ]
 ]
)


عدد الأسئلة العربية للمقارنة: 43
عدد الأسئلة الإنجليزية للمقارنة المشتركة: 3
عدد أسئلة المقارنة الكلي: 46
EN025 محفوظ كتقييم إضافي لـen_fixed: 1


,language,evaluation_group,عدد الأسئلة
0,ar,primary_ar_comparison,43
1,en,primary_en_comparison,3


,question_id,language,question,evaluation_status,gold_reference_in_chunk_corpus
34,AR035,ar,ما المقصود باتفاقية مشاركة البيانات؟,needs_section_mapping,section_not_mapped
84,EN025,en,What rights do individuals have under the HRSD...,needs_section_mapping,Yes


In [34]:
# فحص نوع وقيم الأهلية وحالة الأسئلة قبل اعتماد مجموعة التقييم

print("نوع حقل الأهلية:", gold_df["eligible_for_article_retrieval_eval"].dtype)
print("قيم الأهلية الفعلية:")
display(
 gold_df["eligible_for_article_retrieval_eval"]
 .astype(str)
 .value_counts(dropna=False)
 .rename_axis("القيمة")
 .reset_index(name="العدد")
)

print("توزيع الحالات حسب اللغة:")
display(pd.crosstab(
 gold_df["language"],
 gold_df["evaluation_status"],
 margins=True
))

print("أعداد provisional حسب اللغة:")
display(
 gold_df.loc[
 gold_df["evaluation_status"].eq("provisional_article_match")
 ]
 .groupby("language")["question_id"]
 .nunique()
 .rename("عدد provisional")
 .reset_index()
)

# الأسئلة العربية ذات دليل كامل في المجموعتين
ar_verified_ids = set(
 arabic_document_audit.loc[
 arabic_document_audit["dataset"].isin(["ar_fixed", "ar_article"])
 & arabic_document_audit["evidence_found_in_combined_document"],
 "question_id"
 ].astype(str)
)

# معاينة معرفات provisional والتداخل مع المطابقة العربية
ar_provisional_all = set(
 gold_df.loc[
 gold_df["language"].eq("ar")
 & gold_df["evaluation_status"].eq("provisional_article_match"),
 "question_id"
 ].astype(str)
)

en_provisional_all = set(
 gold_df.loc[
 gold_df["language"].eq("en")
 & gold_df["evaluation_status"].eq("provisional_article_match"),
 "question_id"
 ].astype(str)
)

print("العربية — provisional:", len(ar_provisional_all))
print("العربية — دليل متحقق في المستند:", len(ar_verified_ids))
print("العربية — تقاطع المجموعتين:", len(ar_provisional_all & ar_verified_ids))
print("الإنجليزية — provisional:", len(en_provisional_all))
print("معرفات الإنجليزية provisional:", sorted(en_provisional_all))


نوع حقل الأهلية: bool
قيم الأهلية الفعلية:


,القيمة,العدد
0,False,58
1,True,32


توزيع الحالات حسب اللغة:


evaluation_status,abstention_only,needs_section_mapping,out_of_corpus,provisional_article_match,unanswerable,All
language,,,,,,
ar,10,13,5,32,0,60
en,0,4,0,21,5,30
All,10,17,5,53,5,90


أعداد provisional حسب اللغة:


,language,عدد provisional
0,ar,32
1,en,21


العربية — provisional: 32
العربية — دليل متحقق في المستند: 11
العربية — تقاطع المجموعتين: 0
الإنجليزية — provisional: 21
معرفات الإنجليزية provisional: ['EN001', 'EN002', 'EN003', 'EN004', 'EN005', 'EN006', 'EN007', 'EN008', 'EN009', 'EN011', 'EN012', 'EN013', 'EN014', 'EN015', 'EN016', 'EN017', 'EN018', 'EN019', 'EN020', 'EN021', 'EN022']


In [35]:
# تثبيت أسئلة المقارنة وفق تصنيف Gold وتقسيم المشروع

# أسئلة المقارنة العربية: 32 provisional و13 بحاجة إلى ربط قسم
ar_ids = set(
 gold_df.loc[
 gold_df["language"].eq("ar")
 & gold_df["evaluation_status"].isin(
 ["provisional_article_match", "needs_section_mapping"]
 ),
 "question_id"
 ].astype(str)
)

# أسئلة المقارنة الإنجليزية: 21 provisional وثلاثة مراجع مرتبطة من حالات mapping
en_provisional_ids = set(
 gold_df.loc[
 gold_df["language"].eq("en")
 & gold_df["evaluation_status"].eq("provisional_article_match"),
 "question_id"
 ].astype(str)
)
en_ids = en_provisional_ids | {"EN010", "EN023", "EN024"}

# التحقق من أعداد المجموعة قبل حساب المقاييس
assert len(ar_ids) == 45, f"المتوقع 45 سؤالًا عربيًا، الموجود {len(ar_ids)}"
assert len(en_ids) == 24, f"المتوقع 24 سؤالًا إنجليزيًا، الموجود {len(en_ids)}"

# إنشاء مقارنة داخل اللغة نفسها بين Fixed-size وArticle-level
paired_eval = pd.concat([
 gold_df.loc[gold_df["question_id"].astype(str).isin(ar_ids)].assign(
 dataset="ar_fixed", comparison_scope="paired"
 ),
 gold_df.loc[gold_df["question_id"].astype(str).isin(ar_ids)].assign(
 dataset="ar_article", comparison_scope="paired"
 ),
 gold_df.loc[gold_df["question_id"].astype(str).isin(en_ids)].assign(
 dataset="en_fixed", comparison_scope="paired"
 ),
 gold_df.loc[gold_df["question_id"].astype(str).isin(en_ids)].assign(
 dataset="en_article", comparison_scope="paired"
 ),
], ignore_index=True)

# إبقاء EN025 كتقييم إضافي لـFixed-size فقط
en025_extra = gold_df.loc[
 gold_df["question_id"].astype(str).eq("EN025")
].assign(dataset="en_fixed", comparison_scope="fixed_only_extra")

evaluation_cohort = pd.concat(
 [paired_eval, en025_extra],
 ignore_index=True
)

# توضيح أن ربط AR035 يحتاج مراجعة يدوية قبل قياسه آليًا
evaluation_cohort["reference_mapping_note"] = ""
evaluation_cohort.loc[
 evaluation_cohort["question_id"].astype(str).eq("AR035"),
 "reference_mapping_note"
] = "مراجعة يدوية مطلوبة: gold evidence جزئي وغير مربوط بمقطع محدد"

# عرض تحقق الأعداد
cohort_counts = (
 paired_eval.groupby(["dataset", "language"])["question_id"]
 .nunique()
 .rename("عدد الأسئلة")
 .reset_index()
)

display(cohort_counts)
print("إجمالي أسئلة المقارنة المشتركة:", len(ar_ids) + len(en_ids))
print("EN025 محفوظ كتقييم إضافي في en_fixed:", len(en025_extra))
print("AR035 محفوظ ضمن cohort، لكنه يحتاج ربطًا يدويًا قبل احتسابه آليًا.")


,dataset,language,عدد الأسئلة
0,ar_article,ar,45
1,ar_fixed,ar,45
2,en_article,en,24
3,en_fixed,en,24


إجمالي أسئلة المقارنة المشتركة: 69
EN025 محفوظ كتقييم إضافي في en_fixed: 1
AR035 محفوظ ضمن cohort، لكنه يحتاج ربطًا يدويًا قبل احتسابه آليًا.


In [36]:
# ربط أسئلة المقارنة بالمقاطع الذهبية الفعلية قبل حساب المقاييس
import re
import unicodedata
import numpy as np
import pandas as pd

# توحيد نص الدليل للبحث الحرفي بعد إزالة الترقيم والمسافات الزائدة
def normalize_evidence(text):
 text = unicodedata.normalize("NFKC", str(text or "")).casefold()
 text = re.sub(r"[\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)
 text = text.replace("ـ", "").replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
 return re.sub(r"\s+", " ", re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)).strip()

# تجهيز الأسئلة ومجموعات التقييم، مع إضافة EN025 لـen_fixed فقط
eval_pairs = paired_eval[["question_id", "dataset"]].drop_duplicates()
eval_pairs = pd.concat([
 eval_pairs,
 pd.DataFrame([{"question_id": "EN025", "dataset": "en_fixed"}])
], ignore_index=True).drop_duplicates()

gold_fields = gold_df[[
 "question_id", "gold_document_id", "gold_article_number",
 "gold_evidence_extract", "evaluation_status"
]].copy()

# تجهيز أزواج المستند/المادة من المجموعات، مع covered_article_numbers
chunk_refs = chunks_df[
 ["dataset", "chunk_id", "document_id", "article_number"]
].copy()
chunk_refs["article_value"] = chunk_refs["article_number"]

covered_refs = chunks_df[
 ["dataset", "chunk_id", "document_id", "covered_article_numbers"]
].copy().explode("covered_article_numbers")
covered_refs["article_value"] = covered_refs["covered_article_numbers"]

chunk_refs = pd.concat([
 chunk_refs[["dataset", "chunk_id", "document_id", "article_value"]],
 covered_refs[["dataset", "chunk_id", "document_id", "article_value"]],
], ignore_index=True)

# تطبيع مفاتيح المستند والمادة للمطابقة الدقيقة
chunk_refs["doc_key"] = chunk_refs["document_id"].astype(str).str.strip().str.casefold()
chunk_refs["article_key"] = pd.to_numeric(
 chunk_refs["article_value"], errors="coerce"
).astype("Int64").astype("string")

gold_fields["doc_key"] = gold_fields["gold_document_id"].astype(str).str.strip().str.casefold()
gold_fields["article_key"] = pd.to_numeric(
 gold_fields["gold_article_number"], errors="coerce"
).astype("Int64").astype("string")

# المطابقة القانونية بحسب المستند ورقم المادة
article_matches = (
 eval_pairs.merge(gold_fields, on="question_id", how="left")
 .merge(
 chunk_refs,
 on=["dataset", "doc_key", "article_key"],
 how="inner"
 )
)
article_matches["match_rule"] = "document_id + article_number"

# مطابقة مقتطف الدليل كاملًا داخل المقطع للحالات التي تحتاج ربط قسم
evidence_candidates = (
 eval_pairs.merge(gold_fields, on="question_id", how="left")
 .merge(
 chunks_df[["dataset", "chunk_id", "document_id", "display_text"]],
 left_on=["dataset", "doc_key"],
 right_on=["dataset", "document_id"],
 how="inner"
 )
)
evidence_candidates["evidence_norm"] = evidence_candidates[
 "gold_evidence_extract"
].map(normalize_evidence)
evidence_candidates["text_norm"] = evidence_candidates[
 "display_text"
].map(normalize_evidence)

evidence_matches = evidence_candidates.loc[
 evidence_candidates["evidence_norm"].ne("")
 & evidence_candidates.apply(
 lambda row: row["evidence_norm"] in row["text_norm"], axis=1
 )
].copy()
evidence_matches["match_rule"] = "full_gold_evidence_in_chunk"

# دمج طرق الربط وإزالة التكرارات دون حذف المقاطع
gold_chunk_map = pd.concat(
 [
 article_matches[
 ["question_id", "dataset", "chunk_id", "match_rule"]
 ],
 evidence_matches[
 ["question_id", "dataset", "chunk_id", "match_rule"]
 ],
 ],
 ignore_index=True
).drop_duplicates(["question_id", "dataset", "chunk_id"])

# تقرير يوضح ما أمكن ربطه وما يحتاج مراجعة
mapping_audit = eval_pairs.merge(
 gold_chunk_map.groupby(["question_id", "dataset"]).agg(
 relevant_chunk_count=("chunk_id", "nunique"),
 relevant_chunk_ids=("chunk_id", lambda values: ", ".join(sorted(set(values)))),
 matching_rules=("match_rule", lambda values: ", ".join(sorted(set(values)))),
 ).reset_index(),
 on=["question_id", "dataset"],
 how="left"
)

mapping_audit["relevant_chunk_count"] = (
 mapping_audit["relevant_chunk_count"].fillna(0).astype(int)
)
mapping_audit["mapping_status"] = np.where(
 mapping_audit["relevant_chunk_count"].gt(0),
 "resolved",
 "unresolved_manual_review"
)

print("حالة ربط المراجع حسب المجموعة:")
display(
 mapping_audit.groupby(["dataset", "mapping_status"])
 .size().rename("عدد الأسئلة").reset_index()
)

print("الحالات غير المحسومة:")
display(
 mapping_audit.loc[
 mapping_audit["mapping_status"].eq("unresolved_manual_review")
 ].sort_values(["dataset", "question_id"])
)

print("عينة من جدول الربط:")
display(mapping_audit.head(12))


حالة ربط المراجع حسب المجموعة:


,dataset,mapping_status,عدد الأسئلة
0,ar_article,resolved,45
1,ar_fixed,resolved,45
2,en_article,resolved,24
3,en_fixed,resolved,25


الحالات غير المحسومة:


,question_id,dataset,relevant_chunk_count,relevant_chunk_ids,matching_rules,mapping_status


عينة من جدول الربط:


,question_id,dataset,relevant_chunk_count,relevant_chunk_ids,matching_rules,mapping_status
0,AR001,ar_fixed,4,"pdpl_ar_001__fixed_size__001, pdpl_ar_001__fix...",document_id + article_number,resolved
1,AR002,ar_fixed,4,"pdpl_ar_001__fixed_size__001, pdpl_ar_001__fix...",document_id + article_number,resolved
2,AR003,ar_fixed,4,"pdpl_ar_001__fixed_size__001, pdpl_ar_001__fix...",document_id + article_number,resolved
3,AR004,ar_fixed,4,"pdpl_ar_001__fixed_size__001, pdpl_ar_001__fix...",document_id + article_number,resolved
4,AR005,ar_fixed,1,pdpl_ar_002__fixed_size__001,document_id + article_number,resolved
5,AR006,ar_fixed,1,pdpl_ar_004__fixed_size__001,document_id + article_number,resolved
6,AR007,ar_fixed,1,pdpl_ar_005__fixed_size__001,document_id + article_number,resolved
7,AR008,ar_fixed,1,pdpl_ar_006__fixed_size__001,document_id + article_number,resolved
8,AR009,ar_fixed,1,pdpl_ar_012__fixed_size__001,document_id + article_number,resolved
9,AR010,ar_fixed,1,pdpl_ar_018__fixed_size__001,document_id + article_number,resolved


In [37]:
# تصحيح المطابقة: منع ربط أرقام المواد المفقودة ببعضها
valid_gold = gold_fields.dropna(subset=["article_key"])
valid_chunks = chunk_refs.dropna(subset=["article_key"])

# استخدام المستند والمادة للأسئلة ذات المراجع المرقمة المبدئية
article_matches = (
 eval_pairs
 .merge(
 valid_gold.loc[
 valid_gold["evaluation_status"].eq("provisional_article_match")
 ],
 on="question_id",
 how="inner"
 )
 .merge(
 valid_chunks,
 on=["dataset", "doc_key", "article_key"],
 how="inner"
 )
)
article_matches["match_rule"] = "document_id + article_number"

# استخدام الدليل الكامل للأسئلة التي تحتاج ربط قسم، ومنها EN025
section_matches = evidence_matches.loc[
 evidence_matches["evaluation_status"].eq("needs_section_mapping")
].copy()
section_matches["match_rule"] = "full_gold_evidence_in_chunk"

# إعادة إنشاء المراجع الصحيحة دون المطابقات الناتجة عن القيم المفقودة
columns = ["question_id", "dataset", "chunk_id", "match_rule"]

gold_chunk_map = pd.concat(
 [article_matches[columns], section_matches[columns]],
 ignore_index=True
).drop_duplicates(["question_id", "dataset", "chunk_id"])

# إنشاء تقرير الربط المصحح
resolved_refs = (
 gold_chunk_map.groupby(["question_id", "dataset"])
 .agg(
 relevant_chunk_count=("chunk_id", "nunique"),
 relevant_chunk_ids=("chunk_id", lambda ids: ", ".join(sorted(set(ids)))),
 matching_rules=("match_rule", lambda rules: ", ".join(sorted(set(rules)))),
 )
 .reset_index()
)

mapping_audit = eval_pairs.merge(
 resolved_refs, on=["question_id", "dataset"], how="left"
)
mapping_audit["relevant_chunk_count"] = (
 mapping_audit["relevant_chunk_count"].fillna(0).astype(int)
)
mapping_audit["mapping_status"] = np.where(
 mapping_audit["relevant_chunk_count"].gt(0),
 "resolved",
 "unresolved_manual_review"
)

# عرض الملخص والحالات غير المحسومة
display(
 mapping_audit.groupby(["dataset", "mapping_status"])
 .size().rename("عدد الأسئلة").reset_index()
)

print("الحالات التي تحتاج مراجعة:")
display(mapping_audit.loc[
 mapping_audit["mapping_status"].eq("unresolved_manual_review")
])

print("التحقق من AR035 وEN025:")
display(mapping_audit.loc[
 mapping_audit["question_id"].isin(["AR035", "EN025"])
])


,dataset,mapping_status,عدد الأسئلة
0,ar_article,resolved,44
1,ar_article,unresolved_manual_review,1
2,ar_fixed,resolved,44
3,ar_fixed,unresolved_manual_review,1
4,en_article,resolved,24
5,en_fixed,resolved,25


الحالات التي تحتاج مراجعة:


,question_id,dataset,relevant_chunk_count,relevant_chunk_ids,matching_rules,mapping_status
34,AR035,ar_fixed,0,NaN,NaN,unresolved_manual_review
79,AR035,ar_article,0,NaN,NaN,unresolved_manual_review


التحقق من AR035 وEN025:


,question_id,dataset,relevant_chunk_count,relevant_chunk_ids,matching_rules,mapping_status
34,AR035,ar_fixed,0,NaN,NaN,unresolved_manual_review
79,AR035,ar_article,0,NaN,NaN,unresolved_manual_review
138,EN025,en_fixed,1,hrsd_780009_en_fixed_0003,full_gold_evidence_in_chunk,resolved


In [38]:
# تقييم Recall@5 وMRR@5 للأسئلة المرتبطة بمقاطع ذهبية

TOP_K = 5
METHODS = ["bm25", "dense", "hybrid_rrf", "hybrid_rerank"]

# إبقاء الأسئلة ذات المراجع المرتبطة فقط
resolved_eval = (
 mapping_audit.loc[mapping_audit["mapping_status"].eq("resolved")]
 .merge(gold_df[["question_id", "question"]], on="question_id", how="left")
 .copy()
)

# تمييز مجموعة المقارنة عن تقييم EN025 الإضافي
resolved_eval["language"] = resolved_eval["dataset"].map({
 "ar_fixed": "ar",
 "ar_article": "ar",
 "en_fixed": "en",
 "en_article": "en",
})
resolved_eval["evaluation_scope"] = np.where(
 resolved_eval["question_id"].eq("EN025"),
 "fixed_only_extra",
 "paired_comparison",
)

# إنشاء مجموعة المقاطع الذهبية لكل سؤال ومجموعة
gold_chunk_sets = {
 (qid, dataset): set(group["chunk_id"].astype(str))
 for (qid, dataset), group in gold_chunk_map.groupby(
 ["question_id", "dataset"]
 )
}

# ربط أسماء الطرق بدوال الاسترجاع
search_functions = {
 "bm25": lambda q, d: bm25_search(q, d, TOP_K),
 "dense": lambda q, d: dense_search(q, d, TOP_K),
 "hybrid_rrf": lambda q, d: hybrid_search(q, d, TOP_K),
 "hybrid_rerank": lambda q, d: rerank_search(
 q, d, top_k=TOP_K, candidate_k=20
 ),
}

# تشغيل البحث مرة لكل سؤال وطريقة، وحفظ النتائج ورتبة أول تطابق ذهبي
evaluation_runs = [
 {
 "question_id": row.question_id,
 "language": row.language,
 "dataset": row.dataset,
 "method": method,
 "evaluation_scope": row.evaluation_scope,
 "gold_ids": gold_chunk_sets[(row.question_id, row.dataset)],
 "results": search_functions[method](row.question, row.dataset),
 }
 for row in resolved_eval.itertuples(index=False)
 for method in METHODS
]

# حساب رتبة أول مقطع ذهبي لكل سؤال وطريقة
per_question_metrics = pd.DataFrame([
 {
 "question_id": run["question_id"],
 "language": run["language"],
 "dataset": run["dataset"],
 "method": run["method"],
 "evaluation_scope": run["evaluation_scope"],
 "hit_at_5": int(any(
 str(result["chunk_id"]) in run["gold_ids"]
 for result in run["results"]
 )),
 "reciprocal_rank_at_5": (
 1 / min(
 result["rank"]
 for result in run["results"]
 if str(result["chunk_id"]) in run["gold_ids"]
 )
 if any(
 str(result["chunk_id"]) in run["gold_ids"]
 for result in run["results"]
 )
 else 0.0
 ),
 }
 for run in evaluation_runs
])

# حفظ تفاصيل كل نتيجة مسترجعة للتحقق والتدقيق
evaluation_detail_df = pd.DataFrame([
 {
 "question_id": run["question_id"],
 "language": run["language"],
 "dataset": run["dataset"],
 "method": run["method"],
 "evaluation_scope": run["evaluation_scope"],
 "rank": result["rank"],
 "chunk_id": result["chunk_id"],
 "is_gold_match": str(result["chunk_id"]) in run["gold_ids"],
 "score": result["score"],
 "text": result["text"],
 }
 for run in evaluation_runs
 for result in run["results"]
])

# حساب المقاييس الأساسية؛ يستثني EN025 لأنه تقييم إضافي لـen_fixed فقط
paired_metrics = per_question_metrics.loc[
 per_question_metrics["evaluation_scope"].eq("paired_comparison")
]

retrieval_metrics_by_dataset = (
 paired_metrics.groupby(["language", "dataset", "method"])
 .agg(
 eligible_questions=("question_id", "nunique"),
 recall_at_5=("hit_at_5", "mean"),
 mrr_at_5=("reciprocal_rank_at_5", "mean"),
 )
 .reset_index()
)

retrieval_metrics_by_language = (
 paired_metrics.groupby(["language", "method"])
 .agg(
 eligible_questions=("question_id", "nunique"),
 recall_at_5=("hit_at_5", "mean"),
 mrr_at_5=("reciprocal_rank_at_5", "mean"),
 )
 .reset_index()
)

# عرض النتائج الأساسية والإضافية منفصلة
display(retrieval_metrics_by_dataset.round(4))
display(retrieval_metrics_by_language.round(4))
display(
 per_question_metrics.loc[
 per_question_metrics["evaluation_scope"].eq("fixed_only_extra")
 ]
)


,language,dataset,method,eligible_questions,recall_at_5,mrr_at_5
0,ar,ar_article,bm25,44,0.4545,0.2811
1,ar,ar_article,dense,44,0.9545,0.6754
2,ar,ar_article,hybrid_rerank,44,0.9091,0.8455
3,ar,ar_article,hybrid_rrf,44,0.6818,0.4337
4,ar,ar_fixed,bm25,44,0.4773,0.3235
5,ar,ar_fixed,dense,44,0.9091,0.7155
6,ar,ar_fixed,hybrid_rerank,44,0.8864,0.8076
7,ar,ar_fixed,hybrid_rrf,44,0.7045,0.4792
8,en,en_article,bm25,24,0.5000,0.3611
9,en,en_article,dense,24,0.9583,0.7285


,language,method,eligible_questions,recall_at_5,mrr_at_5
0,ar,bm25,44,0.4659,0.3023
1,ar,dense,44,0.9318,0.6955
2,ar,hybrid_rerank,44,0.8977,0.8265
3,ar,hybrid_rrf,44,0.6932,0.4564
4,en,bm25,24,0.5208,0.3399
5,en,dense,24,0.9375,0.6542
6,en,hybrid_rerank,24,0.8542,0.7378
7,en,hybrid_rrf,24,0.7292,0.5198


,question_id,language,dataset,method,evaluation_scope,hit_at_5,reciprocal_rank_at_5
544,EN025,en,en_fixed,bm25,fixed_only_extra,0,0.00
545,EN025,en,en_fixed,dense,fixed_only_extra,0,0.00
546,EN025,en,en_fixed,hybrid_rrf,fixed_only_extra,0,0.00
547,EN025,en,en_fixed,hybrid_rerank,fixed_only_extra,1,0.25


In [39]:
# حفظ تقارير النزاهة والمقاييس وأمثلة الاسترجاع والملخص النهائي
import json
import pandas as pd

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# إضافة نص السؤال إلى تفاصيل التقييم
evaluation_detail_export = evaluation_detail_df.merge(
 gold_df[["question_id", "question"]],
 on="question_id",
 how="left"
)

# تجهيز أمثلة الاسترجاع مع بيانات المقطع ومراجع Gold
retrieval_examples_df = pd.DataFrame([
 {
 "question_id": run["question_id"],
 "question": gold_df.loc[
 gold_df["question_id"].eq(run["question_id"]), "question"
 ].iloc[0],
 "language": run["language"],
 "dataset": run["dataset"],
 "method": run["method"],
 "rank": result["rank"],
 "chunk_id": result["chunk_id"],
 "document_id": result.get("document_id"),
 "article_number": result.get("article_number"),
 "score": result["score"],
 "is_gold_match": str(result["chunk_id"]) in run["gold_ids"],
 "source_url": result.get("source_url"),
 "text": result["text"],
 }
 for run in evaluation_runs
 for result in run["results"]
])

# حفظ ملفات CSV المطلوبة
integrity_df.to_csv(OUTPUT_DIR / "dataset_integrity_report.csv", index=False)
retrieval_metrics_by_dataset.to_csv(
 OUTPUT_DIR / "retrieval_metrics_by_dataset.csv", index=False
)
retrieval_metrics_by_language.to_csv(
 OUTPUT_DIR / "retrieval_metrics_by_language.csv", index=False
)
retrieval_examples_df.to_csv(OUTPUT_DIR / "retrieval_examples.csv", index=False)
evaluation_detail_export.to_csv(
 OUTPUT_DIR / "retrieval_evaluation_details.csv", index=False
)

# حفظ خريطة المراجع والأسئلة غير القابلة للإجابة للمراجعة اللاحقة
mapping_audit.to_csv(OUTPUT_DIR / "gold_reference_mapping_audit.csv", index=False)
gold_df.loc[
 gold_df["answerable"].eq(False)
].to_csv(OUTPUT_DIR / "unanswerable_questions_for_E5.csv", index=False)

# تجهيز ملخص JSON بالأعداد والإعدادات والحالات الفعلية
summary = {
 "project": "PDPL Compliance Copilot — Stage 2",
 "data_counts": {
 "chunks_by_dataset": chunks_df.groupby("dataset").size().to_dict(),
 "total_chunks": int(len(chunks_df)),
 "gold_questions": int(len(gold_df)),
 "evaluation_questions_by_dataset": (
 paired_eval.groupby("dataset")["question_id"].nunique().to_dict()
 ),
 "unresolved_reference_cases": (
 mapping_audit.loc[
 mapping_audit["mapping_status"].ne("resolved"),
 ["question_id", "dataset"]
 ].to_dict(orient="records")
 ),
 },
 "models": {
 "embedding": EMBEDDING_MODEL_NAME,
 "reranker": RERANKER_MODEL_NAME,
 "device": DEVICE,
 },
 "methods_evaluated": METHODS,
 "settings": {
 "top_k": TOP_K,
 "rrf_k": RRF_K,
 "rerank_candidate_k": 20,
 },
 "gold_matching": {
 "criteria": [
 "مطابقة معرف المستند ورقم المادة معًا عند توفرهما.",
 "استخدام covered_article_numbers لمجموعة en_fixed.",
 "استخدام مقتطف الدليل الكامل للمراجع غير المرقمة عند تطابقه داخل المقطع.",
 ],
 "AR035": "مستبعد من المقاييس الآلية؛ يحتاج مراجعة ربط المرجع.",
 "EN025": "تقييم إضافي في en_fixed فقط؛ ليس ضمن المقارنة المشتركة.",
 "gold_approved_for_final_eval": bool(
 gold_df["approved_for_final_eval"].fillna(False).any()
 ),
 },
 "metrics_by_dataset": retrieval_metrics_by_dataset.to_dict(orient="records"),
 "metrics_by_language": retrieval_metrics_by_language.to_dict(orient="records"),
 "experiment_status": {
 "retrieval_comparison": "completed_experimental",
 "fixed_vs_article": "completed_within_each_language",
 "cross_lingual": "not_run",
 "generation_and_llm_only": "not_run_no_generation_model",
 "abstention_evaluation": "questions_preserved_not_evaluated",
 },
 "warnings": [
 "نتائج Gold تجريبية؛ approved_for_final_eval=False لجميع الأسئلة.",
 "AR035 يحتاج ربطًا يدويًا قبل إدخاله في مقاييس الاسترجاع.",
 "Cross-lingual وتقييم التوليد والامتناع لم تُنفذ بعد.",
 ],
}

with open(OUTPUT_DIR / "stage2_summary.json", "w", encoding="utf-8") as file:
 json.dump(summary, file, ensure_ascii=False, indent=2, default=str)

# التحقق من إنشاء ملفات التسليم
required_outputs = [
 "dataset_integrity_report.csv",
 "retrieval_metrics_by_dataset.csv",
 "retrieval_metrics_by_language.csv",
 "retrieval_examples.csv",
 "retrieval_evaluation_details.csv",
 "stage2_summary.json",
]

print("ملفات التسليم:")
for name in required_outputs:
 path = OUTPUT_DIR / name
 assert path.exists(), f"الملف غير موجود: {path}"
 print(f"✓ {name} — {path.stat().st_size:,} bytes")


ملفات التسليم:
✓ dataset_integrity_report.csv — 483 bytes
✓ retrieval_metrics_by_dataset.csv — 1,034 bytes
✓ retrieval_metrics_by_language.csv — 469 bytes
✓ retrieval_examples.csv — 4,918,430 bytes
✓ retrieval_evaluation_details.csv — 4,628,668 bytes
✓ stage2_summary.json — 6,842 bytes


In [40]:
# فحص دعم تقييم Cross-lingual بمراجع المستند والمادة
import pandas as pd

# أسئلة المقارنة مع مراجعها الذهبية
questions = (
 paired_eval[["question_id", "language"]]
 .drop_duplicates()
 .merge(
 gold_df[["question_id", "gold_document_id", "gold_article_number"]],
 on="question_id",
 how="left",
 )
)

# البحث عن كل لغة سؤال في مجموعتي اللغة الأخرى
cross_targets = pd.concat([
 questions[questions["language"].eq("ar")].assign(
 dataset="en_fixed", corpus_language="en"
 ),
 questions[questions["language"].eq("ar")].assign(
 dataset="en_article", corpus_language="en"
 ),
 questions[questions["language"].eq("en")].assign(
 dataset="ar_fixed", corpus_language="ar"
 ),
 questions[questions["language"].eq("en")].assign(
 dataset="ar_article", corpus_language="ar"
 ),
], ignore_index=True)

# جمع أرقام المواد من الحقول المباشرة ومن covered_article_numbers
direct_refs = chunks_df[
 ["dataset", "document_id", "article_number"]
].rename(columns={"article_number": "article_value"})

covered_refs = (
 chunks_df[["dataset", "document_id", "covered_article_numbers"]]
 .explode("covered_article_numbers")
 .rename(columns={"covered_article_numbers": "article_value"})
)

corpus_refs = pd.concat([
 direct_refs,
 covered_refs[direct_refs.columns],
], ignore_index=True)

# توحيد مفاتيح المستند ورقم المادة للمطابقة الدقيقة
corpus_refs["doc_key"] = corpus_refs["document_id"].astype("string").str.strip().str.casefold()
corpus_refs["article_key"] = pd.to_numeric(
 corpus_refs["article_value"], errors="coerce"
).astype("Int64").astype("string")

cross_targets["doc_key"] = (
 cross_targets["gold_document_id"].astype("string").str.strip().str.casefold()
)
cross_targets["article_key"] = pd.to_numeric(
 cross_targets["gold_article_number"], errors="coerce"
).astype("Int64").astype("string")

# ربط المراجع بالمقاطع في المجموعة ذات اللغة المقابلة
cross_matches = cross_targets.merge(
 corpus_refs.dropna(subset=["doc_key", "article_key"]),
 on=["dataset", "doc_key", "article_key"],
 how="left",
 indicator=True,
)

# تلخيص عدد الأسئلة الفريدة المدعومة بمراجع فعلية
cross_totals = (
 cross_matches.groupby(["language", "corpus_language", "dataset"])["question_id"]
 .nunique()
 .rename("questions")
)

cross_supported = (
 cross_matches[cross_matches["_merge"].eq("both")]
 .groupby(["language", "corpus_language", "dataset"])["question_id"]
 .nunique()
 .rename("questions_with_reference")
)

crosslingual_support = (
 pd.concat([cross_totals, cross_supported], axis=1)
 .fillna(0)
 .astype(int)
 .reset_index()
)

display(crosslingual_support)


,language,corpus_language,dataset,questions,questions_with_reference
0,ar,en,en_article,45,32
1,ar,en,en_fixed,45,32
2,en,ar,ar_article,24,21
3,en,ar,ar_fixed,24,21


In [41]:
# تقييم الاسترجاع عبر اللغات للمراجع التي أمكن ربطها فقط

# تجهيز مراجع المقاطع، بما فيها المواد المغطاة في en_fixed
direct_refs = chunks_df[
 ["dataset", "chunk_id", "document_id", "article_number"]
].rename(columns={"article_number": "article_value"})

covered_refs = (
 chunks_df[["dataset", "chunk_id", "document_id", "covered_article_numbers"]]
 .explode("covered_article_numbers")
 .rename(columns={"covered_article_numbers": "article_value"})
)

cross_refs = pd.concat(
 [direct_refs, covered_refs[direct_refs.columns]],
 ignore_index=True
)

# إنشاء مفاتيح موحدة للمستند والمادة
cross_refs["doc_key"] = cross_refs["document_id"].astype(str).str.strip().str.casefold()
cross_refs["article_key"] = pd.to_numeric(
 cross_refs["article_value"], errors="coerce"
).astype("Int64").astype("string")

cross_targets["doc_key"] = (
 cross_targets["gold_document_id"].astype(str).str.strip().str.casefold()
)
cross_targets["article_key"] = pd.to_numeric(
 cross_targets["gold_article_number"], errors="coerce"
).astype("Int64").astype("string")

# تحديد المقاطع الذهبية في المجموعة ذات اللغة المقابلة
cross_gold = (
 cross_targets.dropna(subset=["doc_key", "article_key"])
 .merge(
 cross_refs.dropna(subset=["doc_key", "article_key"]),
 on=["dataset", "doc_key", "article_key"],
 how="inner"
 )
 [["question_id", "language", "corpus_language", "dataset", "chunk_id"]]
 .drop_duplicates()
)

# تجهيز الأسئلة ومقاطعها الذهبية
cross_eval = (
 cross_targets[["question_id", "language", "corpus_language", "dataset"]]
 .drop_duplicates()
 .merge(
 gold_df[["question_id", "question"]],
 on="question_id",
 how="left"
 )
 .merge(
 cross_gold.groupby(["question_id", "dataset"])["chunk_id"]
 .apply(lambda ids: set(ids.astype(str)))
 .rename("gold_ids")
 .reset_index(),
 on=["question_id", "dataset"],
 how="inner"
 )
)

# اختيار دالة الاسترجاع
search_functions = {
 "bm25": lambda q, d: bm25_search(q, d, 5),
 "dense": lambda q, d: dense_search(q, d, 5),
 "hybrid_rrf": lambda q, d: hybrid_search(q, d, 5),
 "hybrid_rerank": lambda q, d: rerank_search(q, d, top_k=5, candidate_k=20),
}

# تشغيل الطرق وحساب رتبة أول مرجع صحيح
cross_runs = [
 {
 "question_id": row.question_id,
 "question_language": row.language,
 "corpus_language": row.corpus_language,
 "dataset": row.dataset,
 "method": method,
 "hit": int(any(str(r["chunk_id"]) in row.gold_ids for r in results)),
 "rr": (
 1 / min(
 r["rank"] for r in results
 if str(r["chunk_id"]) in row.gold_ids
 )
 if any(str(r["chunk_id"]) in row.gold_ids for r in results)
 else 0.0
 ),
 }
 for row in cross_eval.itertuples(index=False)
 for method in search_functions
 for results in [search_functions[method](row.question, row.dataset)]
]

crosslingual_metrics = (
 pd.DataFrame(cross_runs)
 .groupby(
 ["question_language", "corpus_language", "dataset", "method"]
 )
 .agg(
 evaluated_questions=("question_id", "nunique"),
 recall_at_5=("hit", "mean"),
 mrr_at_5=("rr", "mean"),
 )
 .reset_index()
)

display(crosslingual_metrics.round(4))


,question_language,corpus_language,dataset,method,evaluated_questions,recall_at_5,mrr_at_5
0,ar,en,en_article,bm25,32,0.1250,0.1250
1,ar,en,en_article,dense,32,0.9375,0.7536
2,ar,en,en_article,hybrid_rerank,32,0.9375,0.7995
3,ar,en,en_article,hybrid_rrf,32,0.4375,0.3031
4,ar,en,en_fixed,bm25,32,0.0938,0.0495
5,ar,en,en_fixed,dense,32,0.8750,0.6833
6,ar,en,en_fixed,hybrid_rerank,32,0.8125,0.6953
7,ar,en,en_fixed,hybrid_rrf,32,0.4062,0.3177
8,en,ar,ar_article,bm25,21,0.0476,0.0476
9,en,ar,ar_article,dense,21,0.7619,0.5532


In [42]:
# حفظ المقاييس المحدّثة وتوثيق استخدام FAISS
import json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

retrieval_metrics_by_dataset.to_csv(
 OUTPUT_DIR / "retrieval_metrics_by_dataset.csv", index=False
)
retrieval_metrics_by_language.to_csv(
 OUTPUT_DIR / "retrieval_metrics_by_language.csv", index=False
)
crosslingual_metrics.to_csv(
 OUTPUT_DIR / "retrieval_metrics_cross_lingual.csv", index=False
)

summary_path = OUTPUT_DIR / "stage2_summary.json"

summary = (
 json.loads(summary_path.read_text(encoding="utf-8"))
 if summary_path.exists()
 else {}
)

summary["vector_search"] = "FAISS IndexFlatIP exact search on normalized E5 embeddings"
summary["faiss_indexes"] = {
 name: int(index.ntotal)
 for name, index in FAISS_INDEXES.items()
}
summary["cross_lingual_metrics"] = json.loads(
 crosslingual_metrics.to_json(orient="records")
)
summary["faiss_status"] = "used_for_dense_retrieval_and_evaluation"

summary_path.write_text(
 json.dumps(summary, ensure_ascii=False, indent=2),
 encoding="utf-8",
)

print("حُفظت المقاييس والملخص المحدّث في:", OUTPUT_DIR)
print("أعداد مقاطع FAISS:", summary["faiss_indexes"])


حُفظت المقاييس والملخص المحدّث في: /kaggle/working/stage2_outputs
أعداد مقاطع FAISS: {'ar_fixed': 241, 'ar_article': 215, 'en_fixed': 181, 'en_article': 275}


In [43]:
# حفظ نتائج Cross-lingual وتحديث ملخص المرحلة
import json

# حفظ جدول Cross-lingual في ملف مستقل
crosslingual_metrics.to_csv(
 OUTPUT_DIR / "retrieval_metrics_cross_lingual.csv",
 index=False
)

# تحميل الملخص السابق وتحديث حالة التجربة
summary_path = OUTPUT_DIR / "stage2_summary.json"

with open(summary_path, "r", encoding="utf-8") as file:
 final_summary = json.load(file)

final_summary["cross_lingual_metrics"] = (
 crosslingual_metrics.to_dict(orient="records")
)
final_summary["experiment_status"]["cross_lingual"] = (
 "completed_on_gold_references_supported_in_opposite_language"
)
final_summary["warnings"].append(
 "مقاييس Cross-lingual تجريبية؛ Gold QA غير معتمدة نهائيًا."
)

# إعادة حفظ الملخص بعد التحديث
with open(summary_path, "w", encoding="utf-8") as file:
 json.dump(final_summary, file, ensure_ascii=False, indent=2, default=str)

# التحقق من وجود الملفات المحدّثة
print("حُفظت نتائج Cross-lingual:")
print(OUTPUT_DIR / "retrieval_metrics_cross_lingual.csv")
print(summary_path)
display(crosslingual_metrics.round(4))


حُفظت نتائج Cross-lingual:
/kaggle/working/stage2_outputs/retrieval_metrics_cross_lingual.csv
/kaggle/working/stage2_outputs/stage2_summary.json


,question_language,corpus_language,dataset,method,evaluated_questions,recall_at_5,mrr_at_5
0,ar,en,en_article,bm25,32,0.1250,0.1250
1,ar,en,en_article,dense,32,0.9375,0.7536
2,ar,en,en_article,hybrid_rerank,32,0.9375,0.7995
3,ar,en,en_article,hybrid_rrf,32,0.4375,0.3031
4,ar,en,en_fixed,bm25,32,0.0938,0.0495
5,ar,en,en_fixed,dense,32,0.8750,0.6833
6,ar,en,en_fixed,hybrid_rerank,32,0.8125,0.6953
7,ar,en,en_fixed,hybrid_rrf,32,0.4062,0.3177
8,en,ar,ar_article,bm25,21,0.0476,0.0476
9,en,ar,ar_article,dense,21,0.7619,0.5532


## خلاصة المرحلة الثانية والنتائج

### ما نُفِّذ
بُنِي نظام استرجاع ثنائي اللغة وقُيِّمت أربع طرق على أسئلة Gold المرتبطة بأدلّة
(44 سؤالًا عربيًّا، 24 إنجليزيًّا)، مع فهارس **FAISS** لكل مجموعة
(ar_article=215، ar_fixed=241، en_article=275، en_fixed=181).

### النتائج (التقطيع حسب المادة — Recall@5 / MRR@5)
| الطريقة | عربي R@5 | عربي MRR | إنجليزي R@5 | إنجليزي MRR |
|---|---|---|---|---|
| BM25 (كلمات مفتاحية) | 0.455 | 0.281 | 0.500 | 0.361 |
| Dense (E5 متعدد اللغات) | **0.955** | 0.675 | **0.958** | 0.729 |
| Hybrid (RRF) | 0.682 | 0.434 | 0.750 | 0.547 |
| Hybrid + Reranker | 0.909 | **0.846** | 0.833 | **0.764** |

### أهم الاستنتاجات
- **الاسترجاع الدلالي ضروري:** Dense (E5) يرفع الاستدعاء من ~0.46–0.50 (BM25) إلى
  ~0.95 — فالتطابق اللفظي وحده لا يكفي للنص القانوني.
- **إعادة الترتيب ترفع جودة الترتيب:** الـ Cross-Encoder يقفز بالـ MRR إلى ~0.76–0.85
  (مقابل ~0.68–0.73 لـ Dense)، أي يضع الدليل الصحيح في المراتب الأولى.
- **الأفضل = هجين + إعادة ترتيب:** يجمع استدعاءً عاليًا وأفضل ترتيب معًا، وهو ما
  **نعتمده** مدخلًا لمرحلة التوليد (المرحلة الثالثة).
- **التقطيع حسب المادة ≥ الثابت:** استدعاء ودقّة أعلى، ويحمل رقم المادة (يتيح
  الاستشهاد) — تأكيدٌ لقرار المرحلة الأولى.
- **الاسترجاع عبر اللغات (نتيجة مفصلية):** بالتضمين متعدد اللغات، سؤال عربي على مدونة
  إنجليزية = **R@5 0.9375**، وسؤال إنجليزي على مدونة عربية = **0.7619**؛ بينما
  **BM25 ينهار عبر اللغات** (~0.05–0.13). أي أن النظام يجيب بلغةٍ مستندًا إلى نصّ
  باللغة الأخرى — قدرة جوهرية لمشكلة تعدّد لغات المصادر.

### المخرجات
حُفِظت في `stage2_outputs/`: تقرير سلامة البيانات، مقاييس حسب المجموعة وحسب اللغة،
أمثلة الاسترجاع، تفاصيل التقييم، مقاييس عبر اللغات، و`stage2_summary.json` —
جاهزة للمرحلة الثالثة (التوليد المؤسَّس).

### القيود
التقييم على الأسئلة المرتبطة بأدلّة مؤكَّدة فقط؛ وأسئلة خارج المدونة تُعالَج لاحقًا
بآلية الامتناع في مرحلة التوليد.